In [1]:
import pandas as pd

df = pd.read_csv("openfoodfacts.csv", sep="\t", low_memory=False, on_bad_lines="skip")

print("Finished")

Finished


In [2]:
import numpy as np
import re
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import unicodedata
from collections import Counter

In [3]:
#Duplikate entfernen

In [4]:
df.info(verbose=True, show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 4501355 entries, 0 to 4501354
Data columns (total 210 columns):
 #    Column                            Non-Null Count    Dtype  
---   ------                            --------------    -----  
 0    code                              4501355 non-null  object 
 1    url                               4501355 non-null  str    
 2    creator                           4501349 non-null  str    
 3    created_t                         4501355 non-null  int64  
 4    created_datetime                  4501355 non-null  str    
 5    last_modified_t                   4501355 non-null  int64  
 6    last_modified_datetime            4501355 non-null  str    
 7    last_modified_by                  4362794 non-null  str    
 8    last_updated_t                    4501160 non-null  float64
 9    last_updated_datetime             4501160 non-null  str    
 10   product_name                      4166210 non-null  str    
 11   abbreviated_product_name         

In [5]:
df['code'] = df['code'].astype(str)

# 2. Deine 9 problematischen Barcodes definieren 
# (WICHTIG: Da die Spalte jetzt Text ist, müssen die Zahlen hier in Anführungszeichen stehen!)
barcodes_loeschen = ["1224100812"," 2000000151609", "2000000151619", "2231401010", "3003400425", "63", "7430500116"] 

# 3. Die Kette läuft jetzt fehlerfrei durch
df = (
    df[~df['code'].isin(barcodes_loeschen)]
    .sort_values(by=['code', 'completeness'], ascending=[True, False])
    .drop_duplicates(subset=['code'], keep='first')
    .sort_index() 
)

In [6]:
#Deniz Bereinigung numerische Spalten (Nährwerte)

In [7]:
import pandas as pd
import numpy as np

print("Starte Feature-Engineering: Asche (Ash) berechnen")
print("=================================================\n")

mineralien_liste = [
    "calcium_100g", "iron_100g", "magnesium_100g", "phosphorus_100g", 
    "potassium_100g", "sodium_100g", "zinc_100g", "copper_100g", 
    "manganese_100g", "chloride_100g"
]
vorhandene_mineralien = [col for col in mineralien_liste if col in df.columns]

if vorhandene_mineralien:
    # Berechne die Asche als Summe der vorhandenen Mineralien.
    # min_count=1 stellt sicher, dass Produkte ohne jegliche Mineralien-Einträge bei NaN bleiben
    df['ash_100g'] = df[vorhandene_mineralien].sum(axis=1, min_count=1)
    
    berechnete_aschen = df['ash_100g'].notna().sum()
    print(f"✅ Asche-Generierung: 'ash_100g' wurde bei {berechnete_aschen} Produkten additiv aus Mineralien erzeugt.\n")
else:
    # Falls gar keine Mineralien existieren, legen wir die Spalte als komplett leer an, 
    # damit der nachfolgende Code nicht abstürzt.
    df['ash_100g'] = np.nan
    print("⚠️ Keine Mineralien gefunden. Leere Spalte 'ash_100g' wurde als Platzhalter angelegt.\n")

Starte Feature-Engineering: Asche (Ash) berechnen

✅ Asche-Generierung: 'ash_100g' wurde bei 1929682 Produkten additiv aus Mineralien erzeugt.



/tmp/ipykernel_33032/308931206.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['ash_100g'] = df[vorhandene_mineralien].sum(axis=1, min_count=1)


In [8]:
import pandas as pd
import numpy as np

print("Starte Phase 0: Daten-Harmonisierung & Strikte Validierung")
print("=========================================================\n")

# Globale Variable für die prozentualen Berechnungen
total_rows = len(df)

# --- 1. VALIDIERUNG & BEREINIGUNG: SALZ & NATRIUM ---
if 'salt_100g' in df.columns and 'sodium_100g' in df.columns:
    
    # Vorab-Null-Healing
    maske_natrium_null = (df['sodium_100g'] == 0) & (df['salt_100g'] != 0)
    anzahl_natrium_null = maske_natrium_null.sum()
    if anzahl_natrium_null > 0:
        df.loc[maske_natrium_null, 'salt_100g'] = 0
        pct_natrium_null = (anzahl_natrium_null / total_rows) * 100 if total_rows else 0
        print(f"✅ Vorab-Healing: Bei {anzahl_natrium_null} Produkten ({pct_natrium_null:.2f}% aller Zeilen) wurde Salz auf 0 korrigiert.")
    
    # 1. Prüfen, wo überhaupt beide Werte vorhanden sind
    both_present_salt = df['salt_100g'].notna() & df['sodium_100g'].notna()
    anzahl_gepruefte_salz_paare = both_present_salt.sum()

    # 2. Regel A: Die STRIKTE Toleranz-Regel
    calculated_salt = df['sodium_100g'] * 2.5
    valid_tolerance_salt = np.isclose(df['salt_100g'], calculated_salt, rtol=0.016, atol=0.0)

    # 3. Regel B: Spezial-Regel für ungesalzene Naturprodukte / Wasser
    valid_special_salt = (df['salt_100g'] == 0) & (df['sodium_100g'] > 0)

    # 4. Kombinieren & 5. Fehler identifizieren
    valid_rows_salt = valid_tolerance_salt | valid_special_salt
    invalid_rows_salt = both_present_salt & ~valid_rows_salt
    affected_count_salt = invalid_rows_salt.sum()

    # 6. Die fehlerhaften Zeilen gnadenlos auf NaN setzen
    df.loc[invalid_rows_salt, ['salt_100g', 'sodium_100g']] = np.nan

    # 7. Reporting (mit Prozenten)
    pct_affected_salt_global = (affected_count_salt / total_rows) * 100 if total_rows else 0
    pct_affected_salt_lokal = (affected_count_salt / anzahl_gepruefte_salz_paare) * 100 if anzahl_gepruefte_salz_paare else 0
    
    print("⚡ Check Salz & Natrium abgeschlossen:")
    print(f"   -> Widersprüchliche Zeilen auf NaN gesetzt: {affected_count_salt} ({pct_affected_salt_global:.2f}% aller Zeilen / {pct_affected_salt_lokal:.2f}% der überprüften Paare)")
    print("   -> Grund: Werte lagen außerhalb der strengen Toleranz von 1,6 %.")
    print("   -> Info: Die Spalte 'sodium_100g' wurde plangemäß beibehalten.\n")
else:
    print("Aktion übersprungen: Die Spalten für Salz und Natrium existieren nicht (mehr).\n")


# --- 2. HARMONISIERUNG & VALIDIERUNG: ENERGIE (KCAL & KJ) ---
if 'energy-kcal_100g' in df.columns and 'energy_100g' in df.columns:
    
    # Vorab-Zählung für die Erfolgsquote beim Auffüllen
    luecken_vorher_kcal = df['energy-kcal_100g'].isna().sum()
    
    # SCHRITT A: Die Vorab-Korrektur (Pre-Processing)
    # Fall 1: Falsche kcal-Nullen reparieren
    korrektur_maske_kcal = (df['energy-kcal_100g'] == 0) & (df['energy_100g'] > 0)
    reparierte_nullen_kcal = korrektur_maske_kcal.sum()
    df.loc[korrektur_maske_kcal, 'energy-kcal_100g'] = df.loc[korrektur_maske_kcal, 'energy_100g'] / 4.184

    # Fall 2: Falsche kJ-Nullen reparieren
    korrektur_maske_kj = (df['energy_100g'] == 0) & (df['energy-kcal_100g'] > 0)
    reparierte_nullen_kj = korrektur_maske_kj.sum()
    df.loc[korrektur_maske_kj, 'energy_100g'] = df.loc[korrektur_maske_kj, 'energy-kcal_100g'] * 4.184

    # SCHRITT B: Lücken (NaN) auffüllen
    luecken_maske = df['energy-kcal_100g'].isna() & df['energy_100g'].notna()
    aufgefuellte_luecken = luecken_maske.sum()
    df.loc[luecken_maske, 'energy-kcal_100g'] = df.loc[luecken_maske, 'energy_100g'] / 4.184

    # SCHRITT C: Der strikte Hierarchie-Check
    both_energy_present = df['energy-kcal_100g'].notna() & df['energy_100g'].notna()
    anzahl_gepruefte_energie_paare = both_energy_present.sum()
    
    calculated_kcal = df['energy_100g'] / 4.184
    valid_energy_tolerance = np.isclose(df['energy-kcal_100g'], calculated_kcal, rtol=0.046, atol=0.0)
    
    invalid_energy_rows = both_energy_present & ~valid_energy_tolerance
    geloeschte_energie_zeilen = invalid_energy_rows.sum()

    df.loc[invalid_energy_rows, ['energy-kcal_100g', 'energy_100g']] = np.nan

    # Berechnungen für Prozente
    pct_rep_kcal = (reparierte_nullen_kcal / total_rows) * 100 if total_rows else 0
    pct_rep_kj = (reparierte_nullen_kj / total_rows) * 100 if total_rows else 0
    
    pct_aufg_gesamt = (aufgefuellte_luecken / total_rows) * 100 if total_rows else 0
    pct_aufg_lokal = (aufgefuellte_luecken / luecken_vorher_kcal) * 100 if luecken_vorher_kcal else 0
    
    pct_gel_gesamt = (geloeschte_energie_zeilen / total_rows) * 100 if total_rows else 0
    pct_gel_lokal = (geloeschte_energie_zeilen / anzahl_gepruefte_energie_paare) * 100 if anzahl_gepruefte_energie_paare else 0

    # Reporting
    print("⚡ Check Energie (kcal & kJ) abgeschlossen:")
    print(f"   -> Vorab reparierte kcal-Nullen: {reparierte_nullen_kcal} ({pct_rep_kcal:.2f}% aller Zeilen)")
    print(f"   -> Vorab reparierte kJ-Nullen:   {reparierte_nullen_kj} ({pct_rep_kj:.2f}% aller Zeilen)")
    print(f"   -> Erfolgreich berechnete/aufgefüllte kcal-Lücken: {aufgefuellte_luecken} ({pct_aufg_gesamt:.2f}% aller Zeilen / {pct_aufg_lokal:.1f}% der anfänglichen Lücken)")
    print(f"   -> Widersprüchliche Zeilen auf NaN gesetzt: {geloeschte_energie_zeilen} ({pct_gel_gesamt:.2f}% aller Zeilen / {pct_gel_lokal:.2f}% der überprüften Paare)")
    print("   -> Grund: Werte lagen außerhalb der physikalischen Toleranz von 4,6 %.\n")
else:
    print("Aktion übersprungen: Die Spalten für Energie existieren nicht (mehr).\n")
    
print("-" * 57)
print("🏁 Phase 0 komplett abgeschlossen!")

Starte Phase 0: Daten-Harmonisierung & Strikte Validierung

✅ Vorab-Healing: Bei 722 Produkten (0.02% aller Zeilen) wurde Salz auf 0 korrigiert.
⚡ Check Salz & Natrium abgeschlossen:
   -> Widersprüchliche Zeilen auf NaN gesetzt: 22917 (0.51% aller Zeilen / 1.19% der überprüften Paare)
   -> Grund: Werte lagen außerhalb der strengen Toleranz von 1,6 %.
   -> Info: Die Spalte 'sodium_100g' wurde plangemäß beibehalten.

⚡ Check Energie (kcal & kJ) abgeschlossen:
   -> Vorab reparierte kcal-Nullen: 6336 (0.14% aller Zeilen)
   -> Vorab reparierte kJ-Nullen:   7883 (0.18% aller Zeilen)
   -> Erfolgreich berechnete/aufgefüllte kcal-Lücken: 970 (0.02% aller Zeilen / 0.0% der anfänglichen Lücken)
   -> Widersprüchliche Zeilen auf NaN gesetzt: 423601 (9.41% aller Zeilen / 19.24% der überprüften Paare)
   -> Grund: Werte lagen außerhalb der physikalischen Toleranz von 4,6 %.

---------------------------------------------------------
🏁 Phase 0 komplett abgeschlossen!


In [9]:
import pandas as pd
import numpy as np

print("Starte Phase 1: Physikalische Grenzen (Whitelist-Ansatz)")
print("========================================================\n")

# Globale Telemetrie-Variablen
total_rows = len(df)
gesamte_aenderungen_phase1 = 0
gesamte_vorhandene_werte_phase1 = 0

# --- 1. DEINE LISTEN DEFINIEREN ---
standard_100g = [
       'fat_100g', 'saturated-fat_100g', 'butyric-acid_100g', 'caproic-acid_100g', 'caprylic-acid_100g',
       'capric-acid_100g', 'lauric-acid_100g', 'myristic-acid_100g', 'palmitic-acid_100g', 'stearic-acid_100g', 
       'arachidic-acid_100g', 'behenic-acid_100g', 'lignoceric-acid_100g', 'cerotic-acid_100g',
       'montanic-acid_100g', 'melissic-acid_100g', 'unsaturated-fat_100g', 'monounsaturated-fat_100g', 
       'omega-9-fat_100g', 'polyunsaturated-fat_100g', 'omega-3-fat_100g', 'omega-6-fat_100g',
       'alpha-linolenic-acid_100g', 'eicosapentaenoic-acid_100g', 'docosahexaenoic-acid_100g', 'linoleic-acid_100g',
       'arachidonic-acid_100g', 'gamma-linolenic-acid_100g', 'dihomo-gamma-linolenic-acid_100g', 'oleic-acid_100g',
       'elaidic-acid_100g', 'gondoic-acid_100g', 'mead-acid_100g', 'erucic-acid_100g', 'nervonic-acid_100g', 
       'trans-fat_100g', 'cholesterol_100g', 'carbohydrates_100g', 'sugars_100g', 'added-sugars_100g', 
       'sucrose_100g', 'glucose_100g', 'fructose_100g', 'galactose_100g', 'lactose_100g', 'maltose_100g', 
       'maltodextrins_100g', 'psicose_100g', 'starch_100g', 'polyols_100g', 'erythritol_100g', 'isomalt_100g', 
       'maltitol_100g', 'sorbitol_100g', 'fiber_100g', 'soluble-fiber_100g', 'polydextrose_100g', 
       'insoluble-fiber_100g', 'proteins_100g', 'casein_100g', 'serum-proteins_100g', 'nucleotides_100g', 
       'salt_100g', 'added-salt_100g', 'sodium_100g', 'alcohol_100g', 'vitamin-a_100g', 'beta-carotene_100g',
       'vitamin-d_100g', 'vitamin-e_100g', 'vitamin-k_100g', 'vitamin-c_100g', 'vitamin-b1_100g', 
       'vitamin-b2_100g', 'vitamin-pp_100g', 'vitamin-b6_100g', 'vitamin-b9_100g', 'folates_100g',
       'vitamin-b12_100g', 'biotin_100g', 'pantothenic-acid_100g', 'silica_100g', 'bicarbonate_100g', 
       'potassium_100g', 'chloride_100g', 'calcium_100g', 'phosphorus_100g', 'iron_100g', 'magnesium_100g',
       'zinc_100g', 'copper_100g', 'manganese_100g', 'fluoride_100g', 'selenium_100g', 'chromium_100g', 
       'molybdenum_100g', 'iodine_100g', 'caffeine_100g', 'taurine_100g', 'methylsulfonylmethane_100g',
       'fruits-vegetables-legumes_100g', 'collagen-meat-protein-ratio_100g', 'cocoa_100g', 'chlorophyl_100g',
       'choline_100g', 'phylloquinone_100g', 'beta-glucan_100g', 'inositol_100g', 'carnitine_100g', 
       'sulphate_100g', 'nitrate_100g', 'acidity_100g', 'carbohydrates-total_100g', 'water_100g', 'glycemic-index_100g', "ash_100g"
]

energie_kcal = ['energy-kcal_100g']
energie_kj = ["energy_100g"]
ph_wert = ['ph_100g']
kein_oberes_limit = ['carbon-footprint_100g', 'water-hardness_100g']


# --- 2. DIE FILTER-FUNKTION MIT TELEMETRIE DEFINIEREN ---
def bereinige_gruppe(spalten_liste, limit_unten, limit_oben, gruppen_name):
    global gesamte_aenderungen_phase1, gesamte_vorhandene_werte_phase1
    
    gruppe_falsche_werte = 0
    gruppe_vorhandene_werte = 0
    
    # Schleife über die Spalten der Gruppe
    for spalte in spalten_liste:
        if spalte in df.columns:
            werte_vorher = df[spalte].notna().sum()
            if werte_vorher == 0: 
                continue
            
            # Maske für ungültige Werte anwenden
            unmoeglich = (df[spalte] < limit_unten) | (df[spalte] > limit_oben)
            anzahl_spalte = unmoeglich.sum()
            
            if anzahl_spalte > 0:
                df.loc[unmoeglich, spalte] = np.nan
                gruppe_falsche_werte += anzahl_spalte
            
            gruppe_vorhandene_werte += werte_vorher
            
    # Gruppen-Reporting ausgeben, wenn Spalten Daten enthielten
    if gruppe_vorhandene_werte > 0:
        pct_lokal = (gruppe_falsche_werte / gruppe_vorhandene_werte) * 100
        pct_global_rows = (gruppe_falsche_werte / total_rows) * 100 if total_rows else 0
        
        print(f"⚡ Gruppe '{gruppen_name}' ({limit_unten} bis {limit_oben}):")
        print(f"   -> Falsche Datenpunkte gelöscht (auf NaN): {gruppe_falsche_werte}")
        print(f"   -> Das entspricht {pct_lokal:.2f}% der ausgefüllten Felder in dieser Gruppe.")
        print(f"   -> Betrifft {pct_global_rows:.2f}% aller Produkte im gesamten Datensatz.\n")
    else:
        print(f"📦 Gruppe '{gruppen_name}': Keine Daten in den Spalten vorhanden.\n")
        
    # Werte für das finale Reporting aufsummieren
    gesamte_aenderungen_phase1 += gruppe_falsche_werte
    gesamte_vorhandene_werte_phase1 += gruppe_vorhandene_werte


# --- 3. DIE BEREINIGUNG AUSFÜHREN ---

# Block 1: Deine Riesenliste (0 bis 100)
bereinige_gruppe(standard_100g, 0, 100, "Standard Nährwerte")

# Block 2: kcal (0 bis 900)
bereinige_gruppe(energie_kcal, 0, 900, "Energie (kcal)")

# Block 3: pH-Wert (0 bis 14)
bereinige_gruppe(ph_wert, 0, 14, "pH-Wert")

# Block 4: Nur nach unten begrenzt (0g bis Unendlich)
bereinige_gruppe(kein_oberes_limit, 0, np.inf, "CO2 & Wasserhärte")

# Block 5: kJ (0 bis 3765.6)
bereinige_gruppe(energie_kj, 0, 3765.6, "Energie (kJ)")


# =========================================================
# FINALER PHASE 1 MASTER-REPORT
# =========================================================
print("-" * 60)
print("🏁 ABSCHLUSS-REPORTING PHASE 1 (Physikalische Grenzen):")

if gesamte_vorhandene_werte_phase1 > 0:
    gesamte_pct_lokal = (gesamte_aenderungen_phase1 / gesamte_vorhandene_werte_phase1) * 100
    gesamte_pct_global = (gesamte_aenderungen_phase1 / (total_rows * len(df.columns))) * 100 if total_rows and len(df.columns) else 0
    pct_zeilen_impact = (gesamte_aenderungen_phase1 / total_rows) * 100 if total_rows else 0
    
    print(f"   -> Insgesamt gelöschte fehlerhafte Einzelwerte: {gesamte_aenderungen_phase1}")
    print(f"   -> Fehlerquote über alle geprüften Datenfelder:  {gesamte_pct_lokal:.2f}%")
    print(f"   -> Gesamter Bereinigungs-Impact auf den Datensatz: {pct_zeilen_impact:.2f}% (Verhältnis zu Zeilenanzahl)")
else:
    print("   -> Es wurden keine Daten verarbeitet.")
print("-" * 60)

Starte Phase 1: Physikalische Grenzen (Whitelist-Ansatz)

⚡ Gruppe 'Standard Nährwerte' (0 bis 100):
   -> Falsche Datenpunkte gelöscht (auf NaN): 12445
   -> Das entspricht 0.04% der ausgefüllten Felder in dieser Gruppe.
   -> Betrifft 0.28% aller Produkte im gesamten Datensatz.

⚡ Gruppe 'Energie (kcal)' (0 bis 900):
   -> Falsche Datenpunkte gelöscht (auf NaN): 654
   -> Das entspricht 0.04% der ausgefüllten Felder in dieser Gruppe.
   -> Betrifft 0.01% aller Produkte im gesamten Datensatz.

⚡ Gruppe 'pH-Wert' (0 bis 14):
   -> Falsche Datenpunkte gelöscht (auf NaN): 1
   -> Das entspricht 0.17% der ausgefüllten Felder in dieser Gruppe.
   -> Betrifft 0.00% aller Produkte im gesamten Datensatz.

⚡ Gruppe 'CO2 & Wasserhärte' (0 bis inf):
   -> Falsche Datenpunkte gelöscht (auf NaN): 0
   -> Das entspricht 0.00% der ausgefüllten Felder in dieser Gruppe.
   -> Betrifft 0.00% aller Produkte im gesamten Datensatz.

⚡ Gruppe 'Energie (kJ)' (0 bis 3765.6):
   -> Falsche Datenpunkte gelösch

In [10]:
import pandas as pd
import numpy as np

print("Starte Phase 2: Logische Hierarchien (1-zu-1 Parent-Child Checks)")
print("=================================================================\n")

# Globale Variablen für die Prozentrechnung (Gesamte Zellen im DataFrame)
total_rows = len(df)
total_cols = len(df.columns)
total_zellen_df = total_rows * total_cols

# =========================================================
# DAS ZENTRALE WÖRTERBUCH (Der Stammbaum)
# =========================================================
hierarchie_baum = {
    # 1. Gesamtfett-Familie (Level 1)
    'fat_100g': [
        'saturated-fat_100g',
        'butyric-acid_100g', 'caproic-acid_100g', 'caprylic-acid_100g',
        'capric-acid_100g', 'lauric-acid_100g', 'myristic-acid_100g', 'palmitic-acid_100g', 'stearic-acid_100g',
        'unsaturated-fat_100g', 
        'monounsaturated-fat_100g', 
        'oleic-acid_100g',
        'polyunsaturated-fat_100g', 
        'omega-3-fat_100g', 
        'alpha-linolenic-acid_100g', 'eicosapentaenoic-acid_100g', 'docosahexaenoic-acid_100g', 
        'omega-6-fat_100g',
        'linoleic-acid_100g', 'arachidonic-acid_100g', 'gamma-linolenic-acid_100g',
        'trans-fat_100g', 
        'cholesterol_100g',
    ],
    'saturated-fat_100g': [
        'butyric-acid_100g', 'caproic-acid_100g', 'caprylic-acid_100g', 'capric-acid_100g', 
        'lauric-acid_100g', 'myristic-acid_100g', 'palmitic-acid_100g', 'stearic-acid_100g'
    ],
    'monounsaturated-fat_100g': ['oleic-acid_100g'],
    'polyunsaturated-fat_100g': [
        'omega-3-fat_100g', 'omega-6-fat_100g', 'alpha-linolenic-acid_100g', 'eicosapentaenoic-acid_100g', 
        'docosahexaenoic-acid_100g', 'linoleic-acid_100g', 'arachidonic-acid_100g', 'gamma-linolenic-acid_100g'
    ],
    'omega-3-fat_100g': ['alpha-linolenic-acid_100g', 'eicosapentaenoic-acid_100g', 'docosahexaenoic-acid_100g'],
    'omega-6-fat_100g': ['linoleic-acid_100g', 'arachidonic-acid_100g', 'gamma-linolenic-acid_100g'],
    
    # 2. Kohlenhydrat-Familie (Level 1)
    'carbohydrates_100g': [
        'sugars_100g', 
        'sucrose_100g', 'glucose_100g', 'fructose_100g',
        'galactose_100g', 'lactose_100g', 'maltose_100g','added-sugars_100g', 
        'starch_100g', 
        'polyols_100g', 
        'maltitol_100g', 'sorbitol_100g', 
        'maltodextrins_100g'
    ],
    'sugars_100g': [
        'added-sugars_100g', 'sucrose_100g', 'glucose_100g', 'fructose_100g',
        'galactose_100g', 'lactose_100g', 'maltose_100g'
    ],
    'polyols_100g': ['maltitol_100g', 'sorbitol_100g'],
    
    # 3. Ballaststoff-Familie
    'fiber_100g': ['soluble-fiber_100g', 
                   'beta-glucan_100g',
                   'insoluble-fiber_100g'],
    'soluble-fiber_100g': ['beta-glucan_100g'],
    
    # 4. Protein & Salz
    'proteins_100g': ['casein_100g', 'serum-proteins_100g'],
    'salt_100g': ['added-salt_100g']
}

# Mapping: Welche Obergruppe gehört zu welcher der 5 Hauptfamilien?
familien_mapping = {
    'fat_100g': 'Fett-Familie',
    'saturated-fat_100g': 'Fett-Familie',
    'monounsaturated-fat_100g': 'Fett-Familie',
    'polyunsaturated-fat_100g': 'Fett-Familie',
    'omega-3-fat_100g': 'Fett-Familie',
    'omega-6-fat_100g': 'Fett-Familie',
    'carbohydrates_100g': 'Kohlenhydrat-Familie',
    'sugars_100g': 'Kohlenhydrat-Familie',
    'polyols_100g': 'Kohlenhydrat-Familie',
    'fiber_100g': 'Ballaststoff-Familie',
    'soluble-fiber_100g': 'Ballaststoff-Familie',
    'proteins_100g': 'Protein-Familie',
    'salt_100g': 'Salz-Familie'
}

# Tracking-Dictionary für die Statistik
healing_stats = {
    'Fett-Familie': 0,
    'Kohlenhydrat-Familie': 0,
    'Ballaststoff-Familie': 0,
    'Protein-Familie': 0,
    'Salz-Familie': 0
}


# =========================================================
# SCHRITT 1: DIE NULL-REGEL (Auffüllen / Heilen)
# =========================================================
print("Schritt 1: Führe Null-Healing durch...")
geheilte_null_werte_gesamt = 0

for oberkategorie, unterkategorien in hierarchie_baum.items():
    if oberkategorie in df.columns:
        vorhandene_unterkategorien = [col for col in unterkategorien if col in df.columns]
        
        if vorhandene_unterkategorien:
            maske_null = (df[oberkategorie] == 0)
            anzahl_produkte = maske_null.sum()
            
            if anzahl_produkte > 0:
                # Wir zählen nur noch die echten Änderungen (keine Doppelzählungen)
                echte_aenderungen = (df.loc[maske_null, vorhandene_unterkategorien] != 0).sum().sum()
                
                if echte_aenderungen > 0:
                    geheilte_null_werte_gesamt += echte_aenderungen
                    
                    familie = familien_mapping.get(oberkategorie)
                    if familie:
                        healing_stats[familie] += echte_aenderungen
                    
                    # Daten tatsächlich überschreiben
                    df.loc[maske_null, vorhandene_unterkategorien] = 0

# --- REPORTING FÜR SCHRITT 1 ---
print("\n📊 Report: Echte imputierte 0-Werte nach Hierarchie (ohne Doppelzählungen)")
for familie, anzahl in healing_stats.items():
    relativ_pct = (anzahl / total_zellen_df) * 100 if total_zellen_df > 0 else 0
    print(f"   -> {familie}: {anzahl} Zellen ({relativ_pct:.4f}% aller Zellen im Datensatz)")

relativ_gesamt_pct = (geheilte_null_werte_gesamt / total_zellen_df) * 100 if total_zellen_df > 0 else 0
print(f"\n✅ Healing gesamt: {geheilte_null_werte_gesamt} Zellen wurden bereinigt ({relativ_gesamt_pct:.4f}%).\n")


# =========================================================
# SCHRITT 2: DER HIERARCHIE-FILTER (Löschen auf NaN)
# =========================================================
print("Schritt 2: Führe 1-zu-1 Hierarchie-Prüfung durch...")
geloeschte_hierarchie_werte = 0

for oberkategorie, unterkategorien in hierarchie_baum.items():
    if oberkategorie in df.columns:
        for unterkategorie in unterkategorien:
            if unterkategorie in df.columns:
                
                unmoeglich = df[oberkategorie] < df[unterkategorie]
                anzahl_unmoeglich = unmoeglich.sum()
                
                if anzahl_unmoeglich > 0:
                    df.loc[unmoeglich, unterkategorie] = np.nan
                    geloeschte_hierarchie_werte += anzahl_unmoeglich

print(f"✅ Filter beendet: {geloeschte_hierarchie_werte} unlogische Unter-Werte wurden auf NaN gesetzt.\n")


# =========================================================
# ABSCHLUSS-REPORTING
# =========================================================
print("-" * 55)
print(f"🏁 Phase 2 abgeschlossen!")
print(f"   -> Daten gerettet (0-Imputation): {geheilte_null_werte_gesamt} Zellen")
print(f"   -> Daten gelöscht (NaN-Filter):   {geloeschte_hierarchie_werte} Ausreißer")

Starte Phase 2: Logische Hierarchien (1-zu-1 Parent-Child Checks)

Schritt 1: Führe Null-Healing durch...

📊 Report: Echte imputierte 0-Werte nach Hierarchie (ohne Doppelzählungen)
   -> Fett-Familie: 6035090 Zellen (0.6354% aller Zellen im Datensatz)
   -> Kohlenhydrat-Familie: 2871410 Zellen (0.3023% aller Zellen im Datensatz)
   -> Ballaststoff-Familie: 693693 Zellen (0.0730% aller Zellen im Datensatz)
   -> Protein-Familie: 354577 Zellen (0.0373% aller Zellen im Datensatz)
   -> Salz-Familie: 234446 Zellen (0.0247% aller Zellen im Datensatz)

✅ Healing gesamt: 10189216 Zellen wurden bereinigt (1.0728%).

Schritt 2: Führe 1-zu-1 Hierarchie-Prüfung durch...
✅ Filter beendet: 149577 unlogische Unter-Werte wurden auf NaN gesetzt.

-------------------------------------------------------
🏁 Phase 2 abgeschlossen!
   -> Daten gerettet (0-Imputation): 10189216 Zellen
   -> Daten gelöscht (NaN-Filter):   149577 Ausreißer


In [11]:
import pandas as pd
import numpy as np

print("Starte Phase 3: Mathematische Summenregeln (Massenbilanz)")
print("========================================================\n")

# Globale Variablen für die Prozentrechnung (Gesamte Zellen im DataFrame)
total_rows = len(df)
total_cols = len(df.columns)
total_zellen_df = total_rows * total_cols

geloeschte_werte_phase3 = 0

# Das Dictionary aus Phase 2 - JETZT ERWEITERT UM DIE ASCHE-MINERALIEN
hierarchie_baum = {
    'fat_100g': [
        'saturated-fat_100g', 'butyric-acid_100g', 'caproic-acid_100g', 'caprylic-acid_100g',
        'capric-acid_100g', 'lauric-acid_100g', 'myristic-acid_100g', 'palmitic-acid_100g', 'stearic-acid_100g',
        'unsaturated-fat_100g', 'monounsaturated-fat_100g', 'oleic-acid_100g',
        'polyunsaturated-fat_100g', 'omega-3-fat_100g', 'alpha-linolenic-acid_100g', 
        'eicosapentaenoic-acid_100g', 'docosahexaenoic-acid_100g', 'omega-6-fat_100g',
        'linoleic-acid_100g', 'arachidonic-acid_100g', 'gamma-linolenic-acid_100g',
        'trans-fat_100g', 'cholesterol_100g'
    ],
    'carbohydrates_100g': [
        'sugars_100g', 'sucrose_100g', 'glucose_100g', 'fructose_100g',
        'galactose_100g', 'lactose_100g', 'maltose_100g','added-sugars_100g', 
        'starch_100g', 'polyols_100g', 'maltitol_100g', 'sorbitol_100g', 'maltodextrins_100g'
    ],
    'sugars_100g': [
        'added-sugars_100g', 'sucrose_100g', 'glucose_100g', 'fructose_100g',
        'galactose_100g', 'lactose_100g', 'maltose_100g'
    ],
    'fiber_100g': ['soluble-fiber_100g', 'beta-glucan_100g', 'insoluble-fiber_100g'],
    'proteins_100g': ['casein_100g', 'serum-proteins_100g'],
    'salt_100g': ['added-salt_100g'],
    
    # NEU: Brücke von der Asche zu allen ihren chemischen Untergruppen (Mineralien & Salze)
    'ash_100g': [
        "calcium_100g", "iron_100g", "magnesium_100g", "phosphorus_100g", 
        "potassium_100g", "sodium_100g", "zinc_100g", "copper_100g", 
        "manganese_100g", "chloride_100g"
    ]
}

# --- HELFER-FUNKTION: Alle betroffenen Spalten (inkl. Kinder) sammeln ---
def hole_alle_betroffenen_spalten(start_spalten):
    zu_loeschen = set(start_spalten)
    for spalte in start_spalten:
        if spalte in hierarchie_baum:
            # Füge alle Kinder (die in df existieren) hinzu
            zu_loeschen.update([child for child in hierarchie_baum[spalte] if child in df.columns])
    return list(zu_loeschen)


# =========================================================
# 1. REGEL: FETT-SUMME PRÜFEN
# =========================================================
fett_untergruppen = ['saturated-fat_100g', 'monounsaturated-fat_100g', 'polyunsaturated-fat_100g', 'trans-fat_100g']
if 'fat_100g' in df.columns:
    vorhandene_fett_untergruppen = [col for col in fett_untergruppen if col in df.columns]
    
    if vorhandene_fett_untergruppen:
        summe_untergruppen = df[vorhandene_fett_untergruppen].sum(axis=1)
        bruch_maske = df['fat_100g'].notna() & (df['fat_100g'] < summe_untergruppen)
        anzahl_brueche = bruch_maske.sum()
        
        if anzahl_brueche > 0:
            alle_fett_spalten = hole_alle_betroffenen_spalten(vorhandene_fett_untergruppen)
            geloeschte_zellen = df.loc[bruch_maske, alle_fett_spalten].notna().sum().sum()
            
            df.loc[bruch_maske, alle_fett_spalten] = np.nan
            geloeschte_werte_phase3 += geloeschte_zellen
            
            relativ_zellen = (geloeschte_zellen / total_zellen_df) * 100 if total_zellen_df > 0 else 0
            print(f"🧹 Fett-Summe: {anzahl_brueche}x Brüche korrigiert. {geloeschte_zellen} Zellen (inkl. Sub-Level) auf NaN gesetzt ({relativ_zellen:.4f}% aller Zellen).")


# =========================================================
# 2. REGEL: ZUCKER-SUMME PRÜFEN
# =========================================================
zucker_untergruppen = ['glucose_100g', 'galactose_100g', 'fructose_100g', 'sucrose_100g', 'lactose_100g', 'maltose_100g']
if 'sugars_100g' in df.columns:
    vorhandene_zucker_untergruppen = [col for col in zucker_untergruppen if col in df.columns]
    
    if vorhandene_zucker_untergruppen:
        summe_zucker = df[vorhandene_zucker_untergruppen].sum(axis=1)
        bruch_maske = df['sugars_100g'].notna() & (df['sugars_100g'] < summe_zucker)
        anzahl_brueche = bruch_maske.sum()
        
        if anzahl_brueche > 0:
            alle_zucker_spalten = hole_alle_betroffenen_spalten(vorhandene_zucker_untergruppen)
            geloeschte_zellen = df.loc[bruch_maske, alle_zucker_spalten].notna().sum().sum()
            
            df.loc[bruch_maske, alle_zucker_spalten] = np.nan
            geloeschte_werte_phase3 += geloeschte_zellen
            
            relativ_zellen = (geloeschte_zellen / total_zellen_df) * 100 if total_zellen_df > 0 else 0
            print(f"🧹 Zucker-Summe: {anzahl_brueche}x Brüche korrigiert. {geloeschte_zellen} Zellen auf NaN gesetzt ({relativ_zellen:.4f}% aller Zellen).")


# =========================================================
# 3. REGEL: BALLASTSTOFF-SUMME PRÜFEN
# =========================================================
fiber_untergruppen = ['soluble-fiber_100g', 'insoluble-fiber_100g']
if 'fiber_100g' in df.columns:
    vorhandene_fiber_untergruppen = [col for col in fiber_untergruppen if col in df.columns]
    
    if vorhandene_fiber_untergruppen:
        summe_fiber = df[vorhandene_fiber_untergruppen].sum(axis=1)
        bruch_maske = df['fiber_100g'].notna() & (df['fiber_100g'] < summe_fiber)
        anzahl_brueche = bruch_maske.sum()
        
        if anzahl_brueche > 0:
            alle_fiber_spalten = hole_alle_betroffenen_spalten(vorhandene_fiber_untergruppen)
            geloeschte_zellen = df.loc[bruch_maske, alle_fiber_spalten].notna().sum().sum()
            
            df.loc[bruch_maske, alle_fiber_spalten] = np.nan
            geloeschte_werte_phase3 += geloeschte_zellen
            
            relativ_zellen = (geloeschte_zellen / total_zellen_df) * 100 if total_zellen_df > 0 else 0
            print(f"🧹 Fiber-Summe: {anzahl_brueche}x Brüche korrigiert. {geloeschte_zellen} Zellen (inkl. Sub-Level) auf NaN gesetzt ({relativ_zellen:.4f}% aller Zellen).")


# =========================================================
# 4. REGEL: DER GROSSE 100g PROXIMATE-CHECK (Massenbilanz)
# =========================================================
hauptkomponenten = [
    'water_100g', 'proteins_100g', 'fat_100g', 'carbohydrates_100g', 
    'fiber_100g', 'alcohol_100g', 'ash_100g'
]
vorhandene_hauptkomponenten = [col for col in hauptkomponenten if col in df.columns]

if vorhandene_hauptkomponenten:
    gesamt_masse = df[vorhandene_hauptkomponenten].sum(axis=1, min_count=1)
    
    massen_bruch_maske = gesamt_masse > 100
    anzahl_massen_brueche = massen_bruch_maske.sum()
    
    if anzahl_massen_brueche > 0:
        # AUTOMATISCH: Holt alle Hauptkomponenten + deren gesamten Unterbau (inkl. aller Mineralien der Asche!)
        alle_massen_spalten = hole_alle_betroffenen_spalten(vorhandene_hauptkomponenten)
        
        # Zähle exakt nur die Zellen, die gelöscht werden (ohne bereits leere NaNs mitzuzählen)
        geloeschte_zellen = df.loc[massen_bruch_maske, alle_massen_spalten].notna().sum().sum()
        
        df.loc[massen_bruch_maske, alle_massen_spalten] = np.nan
        geloeschte_werte_phase3 += geloeschte_zellen
        
        relativ_zellen = (geloeschte_zellen / total_zellen_df) * 100 if total_zellen_df > 0 else 0
        print(f"🚨 Massenbilanz-Bruch: {anzahl_massen_brueche} Produkte eliminiert. {geloeschte_zellen} Zellen (gesamte Nährstoff- & Mineralien-Bäume) genullt ({relativ_zellen:.4f}% aller Zellen).")


# =========================================================
# ABSCHLUSS-REPORTING
# =========================================================
relativ_gesamt = (geloeschte_werte_phase3 / total_zellen_df) * 100 if total_zellen_df > 0 else 0

print("-" * 75)
print(f"🏁 Phase 3 abgeschlossen!")
print(f"   -> Insgesamt gelöschte Zellen: {geloeschte_werte_phase3} Zellen (inkl. Unterkategorien & Mineralien)")
print(f"   -> Globaler Impact:            {relativ_gesamt:.4f}% des gesamten Datensatzes")

Starte Phase 3: Mathematische Summenregeln (Massenbilanz)

🧹 Fett-Summe: 2820x Brüche korrigiert. 9899 Zellen (inkl. Sub-Level) auf NaN gesetzt (0.0010% aller Zellen).
🧹 Zucker-Summe: 46980x Brüche korrigiert. 276297 Zellen auf NaN gesetzt (0.0291% aller Zellen).
🧹 Fiber-Summe: 113x Brüche korrigiert. 226 Zellen (inkl. Sub-Level) auf NaN gesetzt (0.0000% aller Zellen).
🚨 Massenbilanz-Bruch: 155593 Produkte eliminiert. 4077726 Zellen (gesamte Nährstoff- & Mineralien-Bäume) genullt (0.4293% aller Zellen).
---------------------------------------------------------------------------
🏁 Phase 3 abgeschlossen!
   -> Insgesamt gelöschte Zellen: 4364148 Zellen (inkl. Unterkategorien & Mineralien)
   -> Globaler Impact:            0.4595% des gesamten Datensatzes


In [12]:
spalten_loeschen = ["energy-kj_100g", "energy-from-fat_100g"]

df = df.drop(columns=spalten_loeschen)

In [13]:
#Allgemeine Funktionen

In [14]:
# ⚙️ SETUP / DEFINITIONEN — verändert df NICHT
# ── Helper ────────────────────────────────────────────────────────────────────
# Junk-Platzhalter -> fehlend. Behaelt bewusst Praefixe wie "en:"/"fr:".
INVALID = {"?", ".", ",", "n-a", "na", "none", "null", "0", "en:null", "en:none"}

def clean_series(s):
    s = s.astype("string").str.strip()
    s = s.replace("", pd.NA)
    s = s.where(~s.str.lower().isin(INVALID), pd.NA)
    s = s.where(~s.str.fullmatch(r"[?,.\-/ ]+", na=False), pd.NA)
    return s

_W = 68

def section_header(title):
    print(f"\n{'═' * _W}")
    print(f"  {title}")
    print(f"{'─' * _W}")

def merge_report(col_name, before, after):
    pct   = after / len(df) * 100
    added = after - before
    bar   = "█" * int(pct / 5) + "░" * (20 - int(pct / 5))
    print(f"  {col_name:<26}  [{bar}]  {pct:5.1f}%   befuellt {after:,}  (+{added:,})")

def clean_report(col_name, before, after):
    pct     = after / len(df) * 100
    removed = before - after
    bar     = "█" * int(pct / 5) + "░" * (20 - int(pct / 5))
    print(f"  {col_name:<26}  [{bar}]  {pct:5.1f}%   befuellt {after:,}  (entfernt {removed:,})")

# Taxonomie-Tags -> lesbare Form, Praefixe ENTFERNT (nur fuer packaging, wie im Hauptnotebook)
def normalize_tags(s):
    return (
        clean_series(s.astype(str).where(s.notna(), np.nan))
        .str.replace(r"\b[a-z]{2}:", "", regex=True)
        .str.replace(r"[-_]", " ", regex=True)
        .str.strip()
        .str.title()
    )


# ── Manufacturing-Places-Helfer: Ländernamen -> Englisch, Regionen/Städte als Eigennamen behalten ──
# Multilinguales Länder-Wörterbuch (Schlüssel deakzentuiert+lowercase) -> englischer Name.
# Regionen/Städte werden NICHT aufs Land kollabiert, zählen aber für die Erkennungs-Quote.
def _deaccent(s):
    return "".join(c for c in unicodedata.normalize("NFKD", str(s)) if not unicodedata.combining(c))

def _mfg_key(tok):
    k = re.sub(r"^[a-z]{2}:", "", _deaccent(tok).lower())
    k = re.sub(r"[-_]", " ", k)
    return re.sub(r"\s+", " ", k).strip()

def _mfg_C(en, *keys):
    return {_mfg_key(k): en for k in keys}

COUNTRY_EN = {}
for _d in [
    _mfg_C("France", "france", "francia", "frankreich", "francie", "frankrijk", "francja"),
    _mfg_C("Germany", "germany", "deutschland", "allemagne", "alemania", "germania", "niemcy", "nemecko", "duitsland", "alemanha"),
    _mfg_C("Switzerland", "switzerland", "suisse", "schweiz", "svizzera", "suiza", "suica", "zwitserland"),
    _mfg_C("Italy", "italy", "italie", "italia", "italien", "wlochy"),
    _mfg_C("Spain", "spain", "espagne", "espana", "spanien", "spagna", "espanha", "hiszpania", "spanje"),
    _mfg_C("Belgium", "belgium", "belgique", "belgie", "belgien", "belgio", "belgia"),
    _mfg_C("Mexico", "mexico", "mexique", "mexiko"),
    _mfg_C("United States", "united states", "usa", "u s a", "etats unis", "estados unidos", "vereinigte staaten", "united states of america", "stati uniti"),
    _mfg_C("United Kingdom", "united kingdom", "uk", "u k", "royaume uni", "reino unido", "grossbritannien", "great britain", "angleterre", "england", "regno unito", "verenigd koninkrijk"),
    _mfg_C("Czech Republic", "czech republic", "cesko", "ceska republika", "tschechien", "republique tcheque", "czechia", "repubblica ceca", "czechy"),
    _mfg_C("Austria", "austria", "osterreich", "autriche", "rakousko", "oostenrijk"),
    _mfg_C("Netherlands", "netherlands", "pays bas", "nederland", "niederlande", "holland", "paesi bassi", "paises bajos"),
    _mfg_C("Poland", "poland", "pologne", "polska", "polen", "polsko"),
    _mfg_C("Tunisia", "tunisia", "tunisie", "tunesien", "tunez"),
    _mfg_C("Norway", "norway", "norvege", "norwegen", "norge", "noruega"),
    _mfg_C("Australia", "australia", "australie", "australien"),
    _mfg_C("Argentina", "argentina", "argentine", "argentinien"),
    _mfg_C("Thailand", "thailand", "thailande", "tailandia"),
    _mfg_C("China", "china", "chine", "cina", "chiny"),
    _mfg_C("Romania", "romania", "roumanie", "rumanien", "rumania"),
    _mfg_C("Portugal", "portugal", "portogallo"),
    _mfg_C("Canada", "canada", "kanada"),
    _mfg_C("Sweden", "sweden", "suede", "schweden", "sverige", "suecia", "szwecja"),
    _mfg_C("Denmark", "denmark", "danemark", "danmark", "dinamarca"),
    _mfg_C("Finland", "finland", "finlande", "suomi", "finlandia", "finnland"),
    _mfg_C("Greece", "greece", "grece", "griechenland", "grecia", "grecja"),
    _mfg_C("Turkey", "turkey", "turquie", "turkei", "turkiye", "turchia", "turquia"),
    _mfg_C("Japan", "japan", "japon", "giappone", "japonia", "japonsko"),
    _mfg_C("India", "india", "inde", "indien"),
    _mfg_C("Brazil", "brazil", "bresil", "brasilien", "brasil", "brasile", "brazylia"),
    _mfg_C("Russia", "russia", "russie", "russland", "rusia", "rosja", "rusko"),
    _mfg_C("Ireland", "ireland", "irlande", "irland", "irlanda"),
    _mfg_C("Hungary", "hungary", "hongrie", "ungarn", "madarsko", "magyarorszag", "hungria"),
    _mfg_C("Slovakia", "slovakia", "slovaquie", "slowakei", "slovensko", "eslovaquia"),
    _mfg_C("Slovenia", "slovenia", "slovenie", "slowenien", "slovenija", "eslovenia"),
    _mfg_C("Croatia", "croatia", "croatie", "kroatien", "hrvatska"),
    _mfg_C("Morocco", "morocco", "maroc", "marokko", "marruecos"),
    _mfg_C("Luxembourg", "luxembourg", "luxemburg", "lussemburgo"),
    _mfg_C("Ukraine", "ukraine", "ukrajina", "ucrania"),
    _mfg_C("New Zealand", "new zealand", "nouvelle zelande", "neuseeland"),
    _mfg_C("South Korea", "south korea", "coree du sud", "korea", "sudkorea", "corea del sur"),
    _mfg_C("Vietnam", "vietnam", "viet nam"),
    _mfg_C("Indonesia", "indonesia", "indonesie", "indonesien"),
    _mfg_C("Serbia", "serbia", "serbie", "srbija"),
    _mfg_C("Bulgaria", "bulgaria", "bulgarie", "bulgarien"),
    _mfg_C("Lithuania", "lithuania", "lituanie", "lietuva", "litauen"),
    _mfg_C("Latvia", "latvia", "lettonie", "letland", "lettland"),
    _mfg_C("Estonia", "estonia", "estonie", "eesti", "estland"),
    _mfg_C("Egypt", "egypt", "egypte", "agypten", "egipto"),
    _mfg_C("Israel", "israel"),
    _mfg_C("Colombia", "colombia", "colombie", "kolumbien"),
    _mfg_C("Chile", "chile", "chili"),
    _mfg_C("Peru", "peru", "perou"),
    _mfg_C("Bolivia", "bolivia", "bolivie", "bolivien"),
    _mfg_C("Ecuador", "ecuador", "equateur"),
    _mfg_C("Uruguay", "uruguay"),
    _mfg_C("Paraguay", "paraguay"),
    _mfg_C("Venezuela", "venezuela"),
    _mfg_C("South Africa", "south africa", "afrique du sud", "sudafrika", "sudafrica"),
    _mfg_C("Iceland", "iceland", "islande", "island", "islandia"),
    _mfg_C("Algeria", "algeria", "algerie", "argelia"),
    _mfg_C("Philippines", "philippines", "filipinas", "filippine"),
    _mfg_C("Singapore", "singapore", "singapour", "singapur"),
    _mfg_C("Malaysia", "malaysia", "malaisie", "malasia"),
    _mfg_C("Sri Lanka", "sri lanka"),
    _mfg_C("Lebanon", "lebanon", "liban", "libano"),
    _mfg_C("Saudi Arabia", "saudi arabia", "arabie saoudite", "arabia saudita"),
    _mfg_C("United Arab Emirates", "united arab emirates", "emirats arabes unis", "uae"),
    _mfg_C("Pakistan", "pakistan"),
    _mfg_C("Iran", "iran"),
    _mfg_C("Cyprus", "cyprus", "chypre", "zypern", "cipro"),
    _mfg_C("Malta", "malta", "malte"),
    _mfg_C("Costa Rica", "costa rica"),
    _mfg_C("Guatemala", "guatemala"),
    _mfg_C("Senegal", "senegal"),
    _mfg_C("Ivory Coast", "ivory coast", "cote d ivoire", "costa de marfil"),
    _mfg_C("Cameroon", "cameroon", "cameroun"),
    _mfg_C("Kenya", "kenya"),
    _mfg_C("Nigeria", "nigeria"),
    _mfg_C("European Union", "european union", "union europeenne", "eu", "u e", "union europea", "europe", "europa"),
]:
    COUNTRY_EN.update(_d)
COUNTRY_EN["россия"] = "Russia"  # häufigste kyrillische Variante

def _mfg_R(country, *keys):
    return {_mfg_key(k): country for k in keys}

# Hochfrequente Regionen/Städte -> Land (nur für die Erkennungs-Quote; Wert bleibt Eigenname)
PLACE_COUNTRY = {}
for _d in [
    _mfg_R("France", "bretagne", "bzh", "normandie", "basse normandie", "haute normandie", "pays de la loire",
       "provence", "provence alpes cote d azur", "alsace", "rhone alpes", "auvergne rhone alpes", "occitanie",
       "bourgogne", "bourgogne franche comte", "aquitaine", "nouvelle aquitaine", "grand est", "hauts de france",
       "ile de france", "nord pas de calais", "centre val de loire", "midi pyrenees", "languedoc roussillon",
       "franche comte", "limousin", "picardie", "champagne ardenne", "lorraine", "poitou charentes", "corse",
       "finistere", "morbihan", "ille et vilaine", "cotes d armor", "loire atlantique", "vendee", "sarthe",
       "maine et loire", "mayenne", "isere", "savoie", "haute savoie", "calvados", "manche", "orne", "nord",
       "loiret", "rhone", "gironde", "herault", "bouches du rhone", "var", "vaucluse", "gard", "drome", "ain",
       "doubs", "jura", "vosges", "moselle", "bas rhin", "haut rhin", "meurthe et moselle", "marne", "aube",
       "yonne", "cote d or", "saone et loire", "puy de dome", "cantal", "allier", "haute loire", "loire",
       "ardeche", "dordogne", "landes", "pyrenees atlantiques", "lot", "aveyron", "tarn", "haute garonne",
       "gers", "aude", "pyrenees orientales", "vienne", "deux sevres", "charente", "charente maritime",
       "indre et loire", "loir et cher", "eure et loir", "eure", "seine maritime", "somme", "pas de calais",
       "aisne", "oise", "ardennes", "meuse", "haute marne", "territoire de belfort", "essonne", "yvelines",
       "val d oise", "seine et marne", "val de marne", "hauts de seine", "seine saint denis",
       "paris", "lyon", "marseille", "bordeaux", "toulouse", "nantes", "lille", "strasbourg", "rennes",
       "quiberon", "durtal", "guadeloupe", "martinique", "la reunion", "reunion", "antilles guyane", "guyane"),
    _mfg_R("Spain", "navarra", "andalucia", "cataluna", "catalunya", "galicia", "murcia", "aragon", "pais vasco",
       "euskadi", "asturias", "cantabria", "la rioja", "castilla y leon", "castilla la mancha", "extremadura",
       "comunidad valenciana", "comunitat valenciana", "canarias", "islas baleares", "islas canarias",
       "madrid", "barcelona", "valencia", "sevilla", "zaragoza", "malaga", "bilbao", "cordoba", "granada",
       "alicante", "gipuzkoa", "bizkaia", "araba", "navarre"),
    _mfg_R("Italy", "lombardia", "piemonte", "toscana", "emilia romagna", "veneto", "sicilia", "campania", "puglia",
       "calabria", "lazio", "liguria", "trentino", "trentino alto adige", "friuli venezia giulia", "marche",
       "abruzzo", "umbria", "basilicata", "molise", "sardegna", "valle d aosta",
       "milano", "roma", "torino", "napoli", "bologna", "firenze", "genova", "parma", "modena", "verona"),
    _mfg_R("Germany", "bayern", "bavaria", "baden wurttemberg", "baden wuerttemberg", "nordrhein westfalen",
       "niedersachsen", "hessen", "sachsen", "rheinland pfalz", "thuringen", "brandenburg", "sachsen anhalt",
       "schleswig holstein", "mecklenburg vorpommern", "saarland", "berlin", "hamburg", "bremen",
       "munchen", "munich", "koln", "frankfurt", "stuttgart", "dusseldorf", "hannover", "nurnberg"),
    _mfg_R("Bolivia", "la paz", "santa cruz", "cochabamba", "el alto", "oruro", "potosi", "sucre", "tarija",
       "santa cruz de la sierra"),
    _mfg_R("Argentina", "buenos aires", "mendoza", "rosario"),
    _mfg_R("Mexico", "ciudad de mexico", "guadalajara", "monterrey", "jalisco"),
    _mfg_R("Switzerland", "geneve", "zurich", "vaud", "valais", "berne", "bern", "ticino"),
    _mfg_R("Belgium", "flandre", "wallonie", "bruxelles", "anvers", "antwerpen"),
    _mfg_R("United Kingdom", "scotland", "wales", "london", "ecosse", "pays de galles"),
    _mfg_R("Portugal", "lisboa", "porto", "madeira", "azores", "acores"),
]:
    PLACE_COUNTRY.update(_d)

_MFG_PREFIX  = re.compile(r"^[a-z]{2}:")
_MFG_SUBKEYS = sorted([k for k in COUNTRY_EN if len(k) >= 4], key=len, reverse=True)
_MFG_SUB_RE  = re.compile(r"\b(" + "|".join(re.escape(k) for k in _MFG_SUBKEYS) + r")\b")

def _mfg_clean_value(val):
    """-> (bereinigter String oder None, erkannt-bool) für EINEN nicht-leeren Wert."""
    parts, seen, recognized = [], set(), False
    for raw in str(val).split(","):
        tok = raw.strip()
        if not tok or tok.lower() in INVALID:
            continue
        k = _mfg_key(tok)
        if not k:
            continue
        if k in COUNTRY_EN:                       # bekanntes Land -> englischer Name
            disp = COUNTRY_EN[k]; recognized = True
        else:                                     # Region/Stadt/Firma -> als Title-Case-Eigenname behalten
            disp = re.sub(r"\s+", " ", re.sub(r"[-_]", " ", _MFG_PREFIX.sub("", tok.lower()))).strip().title()
            if k in PLACE_COUNTRY or _MFG_SUB_RE.search(k):
                recognized = True
        if disp and disp.lower() not in seen:     # je Eintrag case-insensitiv deduplizieren
            seen.add(disp.lower()); parts.append(disp)
    return (",".join(parts), recognized) if parts else (None, False)

def clean_manufacturing(series):
    """Übersetzt Ländernamen -> Englisch, behält Regionen/Städte als Eigennamen.
    Schnell via Unique-Value-Mapping. -> (bereinigte Serie, erkannt-bool-Serie)."""
    s = series.astype("string")
    cmap = {v: _mfg_clean_value(v) for v in pd.unique(s.dropna())}
    cleaned    = s.map(lambda v: cmap[v][0] if v in cmap else pd.NA).astype("string")
    recognized = s.map(lambda v: cmap[v][1] if v in cmap else False).fillna(False).astype(bool)
    return cleaned, recognized


# ── Ozan-Helfer (1:1 portiert aus Datenbereinigung_Ozan NEU; nur das fuer die End-Spalten Noetige) ──
def uebersetze_mit_dict(text, woerterbuch):
    """Trennt am Komma, wendet Woerterbuch an (Key = strip+lower), filtert None, fuegt wieder zusammen."""
    if pd.isna(text):
        return text
    elemente = [woerterbuch.get(land.strip().lower(), land.strip().lower()) for land in str(text).split(",")]
    bereinigt = [land for land in elemente if land is not None]
    if not bereinigt:
        return np.nan
    return ", ".join(bereinigt)

def delete_rows_with_words_in_spalte(df, spalte, woerter):
    return df[~df[spalte].astype(str).str.contains(woerter, na=False, case=False, regex=True)]

def clean_brands_pandas(df, top_n=500):
    df["brands_clean"] = df["brands_en"].str.lower().str.strip()
    df["brands_clean"] = df["brands_clean"].str.normalize("NFKD").str.encode("ascii", errors="ignore").str.decode("utf-8")
    df["brands_clean"] = df["brands_clean"].str.replace(r"[^a-z0-9\s]", " ", regex=True)
    df["brands_clean"] = df["brands_clean"].str.replace(r"\b(gmbh|inc|ltd|sa|co|kg|ag)\b", "", regex=True)
    df["brands_clean"] = df["brands_clean"].str.replace(r"\s+", " ", regex=True).str.strip()
    top_brands = df["brands_clean"].value_counts().nlargest(top_n).index
    df.loc[~df["brands_clean"].isin(top_brands), "brands_clean"] = "other"
    return df

def fill_missing_pnns_groups(df):
    mask_pnns_1 = df["pnns_groups_1"].isna() | (df["pnns_groups_1"] == "unknown")
    mask_pnns_2 = df["pnns_groups_2"].isna() | (df["pnns_groups_2"] == "unknown")
    mask_food   = df["food_groups_en"].notna() & (df["food_groups_en"] != "NaN")
    trigger_condition = mask_pnns_1 & mask_pnns_2 & mask_food
    print(f"  pnns aus food_groups aufgefuellt : {int(trigger_condition.sum()):,} Zeilen")
    if not trigger_condition.any():
        return df
    subset = df.loc[trigger_condition, "food_groups_en"].astype(str)
    split_df = subset.str.rsplit(",", n=1, expand=True)
    if split_df.shape[1] == 1:
        split_df[1] = split_df[0]
    else:
        split_df[1] = split_df[1].fillna(split_df[0])
    pnns_1_clean = split_df[0].str.replace(",", "", regex=False).str.strip()
    pnns_2_clean = split_df[1].str.strip()
    df.loc[trigger_condition, "pnns_groups_1"] = pnns_1_clean
    df.loc[trigger_condition, "pnns_groups_2"] = pnns_2_clean
    return df

def clean_main_category_pandas(df, top_n=200):
    df_cat = df.copy()
    df["main_category_en"] = df["main_category_en"].str.replace(r"^en:", "", regex=True)
    df_cat["main_category_clean"] = df_cat["main_category_en"].str.replace(r"^en:", "", regex=True)
    df_cat["main_category_en"] = df["main_category_en"].str.replace("-", " ").str.strip().str.lower()
    df_cat["main_category_clean"] = df_cat["main_category_clean"].str.replace("-", " ").str.strip().str.lower()
    top_categories = df_cat["main_category_clean"].value_counts().nlargest(top_n).index
    df_cat.loc[~df_cat["main_category_clean"].isin(top_categories), "main_category_clean"] = "other"
    return df_cat


# ── Bogomil-Helfer (1:1 portiert aus Bogomil_Cleaning_main; INVALID -> INVALID_B wg. Namenskollision) ──
# Bogomils INVALID = Memes INVALID zusaetzlich um "" und " " erweitert. Eigener Name, damit das
# globale INVALID (von clean_series in TEIL A-D) NICHT ueberschrieben wird.
INVALID_B = {
    "", " ", "?", ".", ",", "n-a", "na", "none", "null", "0",
    "en:null", "en:none"
}

NON_FOOD_CATEGORIES = {
    'beauty', 'make-up', 'verzorging', 'parfum',
    'indoor and outdoor paints and varnishes',
    'tissue paper and tissue products',
    'hard surface cleaning products',
    'wood- cork- and bamboo-based floor coverings',
    'open beauty facts', 'non food products',
    'nl:beauty', 'nl:nagellakremover', 'nl:nagels', 'nl:nagelverzorging',
    'fr:indoor-and-outdoor-paints-and-varnishes',
    'fr:hard-covering-products',
    'en:non-food-products',
}

# Bekannte fremdsprachige categories_en-Eintraege -> englisches Aequivalent (oder NaN)
CATEGORIES_EN_TRANSLATION = {
    'fr:jus': 'fruit juices',
    'fr:pâtes à tartiner': 'spreads',
    'fr:charcuteries cuites': 'cooked meats',
    'fr:charcuteries diverses': 'cured meats',
    'fr:seves-de-bouleau': np.nan,
    'fr:viander': np.nan,
    'fr:farine-maya': np.nan,
    'nl:beauty': np.nan,
    'nl:nagellakremover': np.nan,
    'nl:nagels': np.nan,
    'nl:nagelverzorging': np.nan,
    'hu:extrudált-kukorica': np.nan,
    'ru:котлета': np.nan,
    'ru:полуфабрикаты': np.nan,
    'fr:indoor-and-outdoor-paints-and-varnishes': np.nan,
    'fr:hard-covering-products': np.nan,
}

# Einzelne, unaufgeloeste fremdsprachige Tags wie 'fr:something'
FOREIGN_PREFIX_PATTERN = re.compile(r'^[a-z]{2}:[a-zA-Z]', re.IGNORECASE)

def clean_categories(value):
    if not isinstance(value, str):
        return np.nan
    value = value.strip().lower()
    if value in INVALID_B:
        return np.nan
    if value in {'undefined', 'en:undefined', 'en:none'}:
        return np.nan
    if value in NON_FOOD_CATEGORIES:
        return np.nan
    if value.replace(',', '').replace(' ', '').isnumeric():
        return np.nan
    if len(value) <= 2:
        return np.nan
    return value

def clean_categories_en(value):
    if not isinstance(value, str):
        return np.nan
    value = value.strip()
    value_lower = value.lower()
    if value_lower in INVALID_B:
        return np.nan
    if value_lower in {'undefined', 'null', 'none'}:
        return np.nan
    if value_lower in NON_FOOD_CATEGORIES:
        return np.nan
    if value_lower.replace(',', '').replace(' ', '').isnumeric():
        return np.nan
    if len(value) <= 2:
        return np.nan
    if value_lower in CATEGORIES_EN_TRANSLATION:
        return CATEGORIES_EN_TRANSLATION[value_lower]
    if ',' not in value and FOREIGN_PREFIX_PATTERN.match(value):
        return np.nan
    return value

def extract_nutriscore_grade(label):
    """Extrahiert Nutri-Score-Note (a-e) aus einem labels_en-String."""
    if not isinstance(label, str):
        return None
    match = re.search(r'nutri.?score[^a-e]*([a-e])', label.lower())
    if match:
        return match.group(1)
    return None

def clean_allergens(value):
    if not isinstance(value, str):
        return np.nan
    value = value.strip().lower()
    allergen_invalid = {
        "", " ", "?", ".", ",", "n-a", "na", "null", "0", "en:null"
    }
    if value in allergen_invalid:
        return np.nan
    if value.replace(',', '').replace(' ', '').isnumeric():
        return np.nan
    return value

def clean_states(value):
    if not isinstance(value, str):
        return np.nan
    return value.strip().lower()

def parse_geo(val):
    try:
        parts = str(val).split(',')
        if len(parts) == 2:
            lat, lon = float(parts[0]), float(parts[1])
            if -90 <= lat <= 90 and -180 <= lon <= 180:
                return val
    except:
        pass
    return np.nan

def clean_data_quality_errors_tags(value):
    if not isinstance(value, str):
        return np.nan
    value = value.strip().lower()
    if value in INVALID_B:
        return np.nan
    return value

EMB_INVALID_VALUES = {
    # Sprach-/Laendercode-Fragmente
    'fr', 'de', 'es', 'it', 'nl', 'pl', 'uk', 'ue', 'eg', 'ec', 'ce',
    'eu', 'be', 'at', 'ch', 'pt', 'se', 'dk', 'fi', 'hu', 'cz', 'sk',
    'ro', 'bg', 'hr', 'si', 'lt', 'lv', 'ee', 'gr', 'ie', 'no', 'y',
    # Platzhalter-Woerter — rohes emb_codes-Format
    'sans', 'absent', 'non', 'neant', 'aucun', 'inconnu', 'maroc',
    'deutschland', 'fao 87', 'no indica', 'not indicated', 'pieces',
    'fabricante', 'envasador', 'otros', 'desconocido', 'comercializador',
    'distribuidor', 'importador',
    # Platzhalter-Woerter — normalisiertes emb_codes_tags-Format
    'sans-estampille', 'not-indicated', 'non-indique', 'non-precise',
    'inexistant', 'neant', 'aucun', 'inconnu',
    'fabricante-y-envasador', 'distribuidor-en-espana',
    'neprecizat', 'perteneciente-a', 'fc-01', 'sif',
}

# Langer Julianischer-Kalender-Satz als Slug
LONG_SLUG_PATTERN = re.compile(r'l-code-l-first-digit', re.IGNORECASE)

# Slugifizierte Firmennamen mit gaengigen Rechtsform-Suffixen
COMPANY_SUFFIX_PATTERN = re.compile(
    r'(-s-a|-s-l|-s-a-s|-s-a-u|-s-l-u|-s-coop|-s-c-a|-gmbh|-ltd|'
    r'-plc|-n-v|-b-v|-s-p-a|-s-r-l|-s-c|-ag|-kg|-inc|-corp|-co|-group'
    r'|-holding|-holdings|-iberica|-espana|-espagne|-france|-italia'
    r'|-deutschland|-manufacturing|-alimentacion|-alimentaria|-alimentarios'
    r'|-productos|-conservas|-aceitunas|-citricos|-agricola|-agroalimentaria'
    r'|-commerciale|-international|-iberia)$',
    re.IGNORECASE
)

def clean_emb_extended(value):
    if not isinstance(value, str):
        return np.nan
    value = value.strip().lower()
    if value in INVALID_B:
        return np.nan
    if value in EMB_INVALID_VALUES:
        return np.nan
    if len(value) <= 3:
        return np.nan
    if value.replace(',', '').replace(' ', '').replace('-', '').isnumeric():
        return np.nan
    if LONG_SLUG_PATTERN.search(value):
        return np.nan
    if COMPANY_SUFFIX_PATTERN.search(value):
        return np.nan
    if re.fullmatch(r'[a-z]?\d{2,4}[a-z]?', value):
        return np.nan
    return value

def remove_en_prefix(value):
    if not isinstance(value, str):
        return value
    tags = [tag.strip() for tag in value.split(',')]
    tags = [tag[3:] if tag.startswith('en:') else tag for tag in tags]
    return ','.join(tags)




print("Finished")

Finished


In [15]:
#Bogomil Bereinigung

In [16]:
df = df.drop(columns=["allergens_en"])
df = df.drop(columns=["states", "states_tags"])

In [17]:
INVALID = {
    "", " ", "?", ".", ",", "n-a", "na", "none", "null", "0",
    "en:null", "en:none"
}

In [18]:
NON_FOOD_CATEGORIES = {
    'beauty', 'make-up', 'verzorging', 'parfum',
    'indoor and outdoor paints and varnishes',
    'tissue paper and tissue products',
    'hard surface cleaning products',
    'wood- cork- and bamboo-based floor coverings',
    'open beauty facts', 'non food products',
    'nl:beauty', 'nl:nagellakremover', 'nl:nagels', 'nl:nagelverzorging',
    'fr:indoor-and-outdoor-paints-and-varnishes',
    'fr:hard-covering-products',
    'en:non-food-products',
}

# Known foreign-language entries in categories_en mapped to English equivalents
CATEGORIES_EN_TRANSLATION = {
    'fr:jus': 'fruit juices',
    'fr:pâtes à tartiner': 'spreads',
    'fr:charcuteries cuites': 'cooked meats',
    'fr:charcuteries diverses': 'cured meats',
    'fr:seves-de-bouleau': np.nan,
    'fr:viander': np.nan,
    'fr:farine-maya': np.nan,
    'nl:beauty': np.nan,
    'nl:nagellakremover': np.nan,
    'nl:nagels': np.nan,
    'nl:nagelverzorging': np.nan,
    'hu:extrudált-kukorica': np.nan,
    'ru:котлета': np.nan,
    'ru:полуфабрикаты': np.nan,
    'fr:indoor-and-outdoor-paints-and-varnishes': np.nan,
    'fr:hard-covering-products': np.nan,
}

# Matches unresolved foreign-prefixed single tags e.g. 'fr:something'
FOREIGN_PREFIX_PATTERN = re.compile(r'^[a-z]{2}:[a-zA-Z]', re.IGNORECASE)

In [19]:
def clean_categories(value):
    if not isinstance(value, str):
        return np.nan
    value = value.strip().lower()
    if value in INVALID:
        return np.nan
    if value in {'undefined', 'en:undefined', 'en:none'}:
        return np.nan
    if value in NON_FOOD_CATEGORIES:
        return np.nan
    if value.replace(',', '').replace(' ', '').isnumeric():
        return np.nan
    if len(value) <= 2:
        return np.nan
    return value


def clean_categories_en(value):
    if not isinstance(value, str):
        return np.nan
    value = value.strip()
    value_lower = value.lower()
    if value_lower in INVALID:
        return np.nan
    if value_lower in {'undefined', 'null', 'none'}:
        return np.nan
    if value_lower in NON_FOOD_CATEGORIES:
        return np.nan
    if value_lower.replace(',', '').replace(' ', '').isnumeric():
        return np.nan
    if len(value) <= 2:
        return np.nan
    if value_lower in CATEGORIES_EN_TRANSLATION:
        return CATEGORIES_EN_TRANSLATION[value_lower]
    # Nullify unresolved single foreign-prefixed tags
    # (comma-separated lists that contain one foreign tag are kept)
    if ',' not in value and FOREIGN_PREFIX_PATTERN.match(value):
        return np.nan
    return value


df['categories'] = df['categories'].apply(clean_categories)
df['categories_en'] = df['categories_en'].apply(clean_categories_en)


In [20]:
df["nutriscore_score"] = pd.to_numeric(df["nutriscore_score"], errors="coerce")

In [21]:
df["nutriscore_grade"] = (
    df["nutriscore_grade"]
    .str.lower()
    .str.strip()
)

df["nutriscore_grade"] = df["nutriscore_grade"].replace({
    "unknown": np.nan,
    "590": np.nan
})

In [22]:
def extract_nutriscore_grade(label):
    """
    Extracts a Nutri-Score grade letter (a-e) from a labels_en string.
    Matches 'nutriscore' or 'nutri-score', skips any intervening text,
    then captures the first grade letter a through e.
    """
    if not isinstance(label, str):
        return None
    match = re.search(r'nutri.?score[^a-e]*([a-e])', label.lower())
    if match:
        return match.group(1)
    return None


extracted = df['labels_en'].apply(extract_nutriscore_grade)

# Only fill rows that are genuinely missing or flagged as not-applicable
# — never overwrite an existing valid grade
fill_mask = df['nutriscore_grade'].isna() | (df['nutriscore_grade'] == 'not-applicable')
valid_extracted = extracted.notna()

df.loc[fill_mask & valid_extracted, 'nutriscore_grade'] = (
    extracted[fill_mask & valid_extracted]
)

In [23]:
def clean_allergens(value):
    if not isinstance(value, str):
        return np.nan
    value = value.strip().lower()
    # Custom invalid set — excludes 'none' and 'en:none' which are valid
    allergen_invalid = {
        "", " ", "?", ".", ",", "n-a", "na", "null", "0", "en:null"
    }
    if value in allergen_invalid:
        return np.nan
    if value.replace(',', '').replace(' ', '').isnumeric():
        return np.nan
    return value


df["allergens"] = df["allergens"].apply(clean_allergens)

In [24]:
def clean_states(value):
    if not isinstance(value, str):
        return np.nan
    return value.strip().lower()


df["states_en"] = df["states_en"].apply(clean_states)

In [25]:
def parse_geo(val):
    try:
        parts = str(val).split(',')
        if len(parts) == 2:
            lat, lon = float(parts[0]), float(parts[1])
            if -90 <= lat <= 90 and -180 <= lon <= 180:
                return val
    except:
        pass
    return np.nan


df['first_packaging_code_geo'] = df['first_packaging_code_geo'].apply(parse_geo)

In [26]:
def clean_data_quality_errors_tags(value):
    if not isinstance(value, str):
        return np.nan
    value = value.strip().lower()
    if value in INVALID:
        return np.nan
    return value


df['data_quality_errors_tags'] = df['data_quality_errors_tags'].apply(
    clean_data_quality_errors_tags
)

In [27]:
EMB_INVALID_VALUES = {
    # Language/country code fragments
    'fr', 'de', 'es', 'it', 'nl', 'pl', 'uk', 'ue', 'eg', 'ec', 'ce',
    'eu', 'be', 'at', 'ch', 'pt', 'se', 'dk', 'fi', 'hu', 'cz', 'sk',
    'ro', 'bg', 'hr', 'si', 'lt', 'lv', 'ee', 'gr', 'ie', 'no', 'y',
    # Placeholder words — raw emb_codes format
    'sans', 'absent', 'non', 'neant', 'aucun', 'inconnu', 'maroc',
    'deutschland', 'fao 87', 'no indica', 'not indicated', 'pieces',
    'fabricante', 'envasador', 'otros', 'desconocido', 'comercializador',
    'distribuidor', 'importador',
    # Placeholder words — normalised emb_codes_tags format
    'sans-estampille', 'not-indicated', 'non-indique', 'non-precise',
    'inexistant', 'neant', 'aucun', 'inconnu',
    'fabricante-y-envasador', 'distribuidor-en-espana',
    'neprecizat', 'perteneciente-a', 'fc-01', 'sif',
}

# Long Julian calendar sentence normalised into a hyphenated slug
LONG_SLUG_PATTERN = re.compile(r'l-code-l-first-digit', re.IGNORECASE)

# Slugified company names end with common legal suffixes
COMPANY_SUFFIX_PATTERN = re.compile(
    r'(-s-a|-s-l|-s-a-s|-s-a-u|-s-l-u|-s-coop|-s-c-a|-gmbh|-ltd|'
    r'-plc|-n-v|-b-v|-s-p-a|-s-r-l|-s-c|-ag|-kg|-inc|-corp|-co|-group'
    r'|-holding|-holdings|-iberica|-espana|-espagne|-france|-italia'
    r'|-deutschland|-manufacturing|-alimentacion|-alimentaria|-alimentarios'
    r'|-productos|-conservas|-aceitunas|-citricos|-agricola|-agroalimentaria'
    r'|-commerciale|-international|-iberia)$',
    re.IGNORECASE
)

In [28]:
def clean_emb_extended(value):
    if not isinstance(value, str):
        return np.nan
    value = value.strip().lower()
    if value in INVALID:
        return np.nan
    if value in EMB_INVALID_VALUES:
        return np.nan
    # Short fragments of 3 characters or fewer
    if len(value) <= 3:
        return np.nan
    # Purely numeric strings — also handles comma-separated numeric combos
    if value.replace(',', '').replace(' ', '').replace('-', '').isnumeric():
        return np.nan
    # Julian calendar sentence slug
    if LONG_SLUG_PATTERN.search(value):
        return np.nan
    # Slugified company names
    if COMPANY_SUFFIX_PATTERN.search(value):
        return np.nan
    # Random short alphanumeric fragments:
    # optional letter + 2–4 digits + optional letter
    # catches 'a879', 'b01h', '1687' but not real codes which contain hyphens
    if re.fullmatch(r'[a-z]?\d{2,4}[a-z]?', value):
        return np.nan
    return value


df['emb_codes'] = df['emb_codes'].apply(clean_emb_extended)
df['emb_codes_tags'] = df['emb_codes_tags'].apply(clean_emb_extended)

In [29]:
def remove_en_prefix(value):
    if not isinstance(value, str):
        return value
    tags = [tag.strip() for tag in value.split(',')]
    tags = [tag[3:] if tag.startswith('en:') else tag for tag in tags]
    return ','.join(tags)


for col in ['allergens', 'states_en', 'data_quality_errors_tags']:
    df[col] = df[col].apply(remove_en_prefix)

In [30]:
#Ozan Bereinigung

In [31]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

# Verhindert den Zeilenumbruch auf mehrere Zeilen
pd.set_option('display.expand_frame_repr', False)

In [32]:
meine_spalten = ["brands", "brands_tags", "brands_en", 
                 "labels", "labels_tags", "labels_en", 
                 "countries", "countries_tags", "countries_en", 
                 "food_groups", "food_groups_tags", "food_groups_en", 
                 "main_category", "main_category_en", 
                 "pnns_groups_1", "pnns_groups_2",
                 "brand_owner",
                 "popularity_tags",
                 "origins", "origins_tags", "origins_en",
                 "purchase_places",
                 "cities_tags",
                 "no_nutrition_data"]

df[meine_spalten] = df[meine_spalten].apply(lambda x: x.strip() if isinstance(x, str) else x)

In [33]:
def completeness_spalte(df, spalte):
    return df[spalte].notna().mean()*100

In [34]:
# ==========================================
# LÖSUNG MIT PANDAS
# ==========================================
def clean_brands_pandas(df, top_n=500):
    # 1. Text normalisieren: Kleinbuchstaben & Whitespaces entfernen
    df['brands_clean'] = df['brands_en'].str.lower().str.strip()
    
    # 2. Akzente entfernen (é -> e, ä -> a)
    # Wichtig: Benötigt oft das 'unidecode' Paket oder Pandas string normalize
    df['brands_clean'] = df['brands_clean'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')
    
    # 3. Satzzeichen und Firmenzusätze (GmbH, Inc, SA) mit Regex entfernen
    # Ersetzt alles was kein Buchstabe oder Zahl ist mit einem Leerzeichen
    df['brands_clean'] = df['brands_clean'].str.replace(r'[^a-z0-9\s]', ' ', regex=True)
    # Entfernt gängige Rechtsformen
    df['brands_clean'] = df['brands_clean'].str.replace(r'\b(gmbh|inc|ltd|sa|co|kg|ag)\b', '', regex=True)
    
    # Letzte doppelte Leerzeichen entfernen, die durch das Löschen entstanden sind
    df['brands_clean'] = df['brands_clean'].str.replace(r'\s+', ' ', regex=True).str.strip()
    
    # 4. THRESHOLDING (Die Top N behalten, den Rest zu 'other' machen)
    # Finde die N häufigsten Marken
    top_brands = df['brands_clean'].value_counts().nlargest(top_n).index
    
    # Alle Marken, die NICHT in den Top Brands sind, werden zu 'other'
    df.loc[~df['brands_clean'].isin(top_brands), 'brands_clean'] = 'other'
    
    return df

In [35]:
# prozent in dezimal angeben, z.B. 0.90 für 90%
# Funktion, die die kumulierten Prozente berechnet und die Einträge zurückgibt, die bis zu einem bestimmten Prozentwert beitragen
def top_x_percent_spalte(df, spalte, prozent, with_lower_case=True):
    if with_lower_case:
        counts = df[spalte].str.split(",").explode().str.strip().str.lower().value_counts()
    else:
        counts = df[spalte].str.split(",").explode().str.strip().value_counts()
    cum_percentage = counts.cumsum() / counts.sum()
    return cum_percentage[cum_percentage <= prozent]

In [36]:
# Löscht die komplette Zeile, wenn "ice cream", "test" ODER "unknown" als eigenständiges Wort vorkommt
#fehler_woerter = r'\b(?:Saba|Snacks|it:ringo-gelato-cacao)\b'
# df = df[~df['countries_en'].str.contains(fehler_woerter, na=False, case=False, regex=True)]

def delete_rows_with_words_in_spalte(df, spalte, woerter):
    return df[~df[spalte].astype(str).str.contains(woerter, na=False, case=False, regex=True)]

In [37]:
def contains_x_in_spalte(df, spalte, x):
    return df[df[spalte].str.contains(x, na=False, case=False, regex=True)]

In [38]:
def top_x_spalte(df, spalte, x):
    return df[spalte].str.split(",").explode().str.strip().value_counts().head(x)

def bottom_x_spalte(df, spalte, x):
    return df[spalte].str.split(",").explode().str.strip().value_counts().tail(x)

In [39]:
def entferne_fehlereintrage(eintrag, fehler_wort_set: set={}):
    # Leere Zellen (NaN) einfach überspringen
    if pd.isna(eintrag):
        return eintrag
        
    # Wenn der Eintrag als kommagetrennter String vorliegt
    if isinstance(eintrag, str):
        # 1. Am Komma trennen und Leerzeichen entfernen
        laender = [land.strip() for land in eintrag.split(',')]
        
        # 2. Alles behalten, was NICHT das Fehlerwort ist (Groß-/Kleinschreibung wird ignoriert)
        bereinigt = [land for land in laender if land.lower() not in fehler_wort_set]
        
        # 3. Wieder mit einem sauberen Komma zusammenfügen
        return ','.join(bereinigt)
        
    # Falls die Einträge im DataFrame durch vorherige Schritte doch als Listen vorliegen
    elif isinstance(eintrag, list):
        return [land for land in eintrag if land.strip().lower() not in fehler_wort_set]
        
    return eintrag

# Die Funktion auf die Spalte anwenden
# df['countries_en'] = df['countries_en'].apply(entferne_fehlereintrag)

In [40]:
def uebersetze_mit_dict(text, woerterbuch):
    """
    Trennt einen String am Komma, wendet ein Wörterbuch an,
    entfernt None-Werte und fügt ihn wieder als String zusammen.
    """
    # Wenn die Zelle komplett leer (NaN) ist, mach nichts
    if pd.isna(text):
        return text
        
    # 1. Am Komma trennen und jedes Element bereinigen (strip + lowercase)
    # 2. Übersetzen: Wenn nicht im Dict, behalte den sauberen Originalwert
    elemente = [woerterbuch.get(land.strip().lower(), land.strip().lower()) for land in str(text).split(',')]
    
    # 3. Das WICHTIGSTE: Alle 'None' Werte herausfiltern, die das .join() zum Abstürzen bringen
    bereinigt = [land for land in elemente if land is not None]
    
    # 4. Wenn die Liste jetzt leer ist (z.B. weil nur "unknown" in der Zelle stand), gib NaN zurück
    if not bereinigt:
        return np.nan
        
    # 5. Als String wieder zusammenfügen (mit Komma und Leerzeichen für die Lesbarkeit)
    return ', '.join(bereinigt)

In [41]:
#Prozent und Anzahl der Spalte vor Bearbeitung
countries_en_completeness = str(df["countries_en"].notna().mean()*100)
countries_en_before = df["countries_en"].notna().sum()

print("Vor Zusammenführen war Spalte zu " + countries_en_completeness + " gefüllt")

#countries_en mit countries_tags auffüllen falls leer, mit countries auffüllen falls auch leer
df["countries_en"] = (df["countries_en"]
                                .fillna(df["countries_tags"]
                                .fillna(df["countries"])))

#Prozent und Anzahl der Spalte nach Bearbeitung
countries_en_completeness = str(df["countries_en"].notna().mean()*100)
countries_en_after = df["countries_en"].notna().sum()

countries_added = str(countries_en_after - countries_en_before)

print("Nach Zusammenführen ist Spalte zu " + countries_en_completeness + " gefüllt")
print("Es wurden " + countries_added + " Einträge hinzugefügt")

#countries und countries_tags löschen
df = df.drop(columns=["countries", "countries_tags"])

Vor Zusammenführen war Spalte zu 99.4892567296924 gefüllt
Nach Zusammenführen ist Spalte zu 99.48936780913354 gefüllt
Es wurden 5 Einträge hinzugefügt


In [42]:
woerterbuch = {
    # --- Deine bisherigen Einträge ---
    'Australien': 'Australia',
    'australien': 'Australia',
    'Deutschland': 'Germany', 
    'Germania': 'Germany', 
    'de:allemagne': 'Germany', 
    'fr:deutschland': 'Germany',
    'Frankreich': 'France', 
    'فرنسا': 'France',
    'fr:italien': 'Italy',
    'Japon': 'Japan',
    'Belgio': 'Belgium',
    'it:paesi-ue': 'EU', 
    'ca:union-europea': 'EU',
    'Suede': 'Sweden',
    'Svizzera': 'Switzerland', 'Schweiz': 'Switzerland',
    'Polonia': 'Poland',
    'صنعاء': 'Yemen',
    'عدن،صنعاء': 'Yemen',
    'Brazil-en-france': 'Brazil, France',
    'de:great-britain': 'United Kingdom',
    'Saudi': 'Saudi Arabia',
    
    # --- Neue Länderzuordnungen ---
    'Irland': 'Ireland',
    'Estados-unidos': 'United States',
    'كندا': 'Canada',
    'British-columbia-canada': 'Canada',
    'Vancouver-bc-canada': 'Canada',
    'Usa-new-york-only': 'United States',
    'St-vincent-and-grenadines': 'Saint Vincent and the Grenadines',
    'فلسطين': 'Palestine',
    'Svalbard and Jan Mayen': 'Svalbard and Jan Mayen',
    'Inde': 'India',
    'Pologne': 'Poland',
    'Roumanie': 'Romania',
    'Turquie': 'Turkey',
    'Algerie': 'Algeria',
    'Polynesie-francaise': 'French Polynesia',
    'de:スペイン': 'Spain',          # Japanisches Katakana für Spanien
    'de:deitschland': 'Germany',   # Dialekt/Tippfehler
    'de:německo': 'Germany',       # Tschechisch für Deutschland
    'nl:roemenie': 'Romania',
    'bg:литвс': 'Lithuania',       # Tippfehler für Litauen (Литва)
    'si:ශ්\u200dරී-ලංකාව': 'Sri Lanka',
    'nl:moldavie': 'Moldova',
    'it:giappo': 'Japan',          # Italienische Abkürzung
    'Wales': 'United Kingdom',     # Teilregion, standardisiert meist UK
    'el:ελλαδα': 'Greece',
    'Saint-Barthélemy': 'Saint Barthélemy',
    'ca:franca': 'France',
    'Eswatini': 'Eswatini',
    'Central-africa': 'Central African Republic',
    'ar:تونس': 'Tunisia',
    'ar:jorsan': 'Jordan',         # Eindeutiger Tippfehler
    'Saba': 'Saba'
}

In [43]:
# Auf die Spalte anwenden
df['countries_en'] = df['countries_en'].apply(lambda x: uebersetze_mit_dict(x, woerterbuch))

In [44]:
# Löscht die komplette Zeile, wenn "ice cream", "test" ODER "unknown" als eigenständiges Wort vorkommt
#fehler_woerter = r'\b(?:Saba|Snacks|it:ringo-gelato-cacao)\b'

#df = df[~df['countries_en'].str.contains(fehler_woerter, na=False, case=False, regex=True)]

In [45]:
# Alles in einem Schritt: Trennen -> Explodieren -> Säubern -> Zählen
laender_counts = (
    df['countries_en']
    .dropna()             # 1. Leere Zellen ignorieren
    .str.split(',')       # 2. Am Komma in Listen aufteilen
    .explode()            # 3. Listen direkt auflösen (passiert nur im Hintergrund)
    .str.strip()          # 4. Störende Leerzeichen vor/nach Ländern entfernen
    .value_counts()       # 5. Alle Länder zählen
)

# Jetzt aus diesem Ergebnis nur die filtern, die exakt 1 mal vorkommen
anzahl = len(laender_counts[laender_counts == 1])
laender_liste = laender_counts[laender_counts <= 50].index.to_list()
print(laender_liste)

print(f"Es gibt {anzahl} Länder, die exakt ein einziges Mal vorkommen.")

['cayman islands', 'laos', 'saint martin', 'antigua and barbuda', 'cape verde', 'francia-espana', 'nl:belgie', 'fr:quebec', 'suisse', 'frankreich', 'grenada', 'malawi', 'tajikistan', 'sint maarten', 'worldwide', 'liberia', 'vanuatu', 'fr:francia', 'chad', 'belgique', 'equatorial guinea', 'ru:nyl', 'deutschland', 'francia', 'anguilla', 'emirats-arabes-unis', 'ประเทศไทย-thai', 'north korea', 'wallis and futuna', 'burundi', 'saint vincent and the grenadines', 'republika srpska', 'allemagne', 'american samoa', 'british virgin islands', 'northern mariana islands', 'gambia', 'vereinigte-staaten-von-amerika', 'state of palestine', 'espagne', 'royaume-uni', 'fr:dom-tom', 'bhutan', 'indonesie', 'mallorca', 'es:cantabria', 'česko', 'angle', 'angleterre', 'irlande', 'liban', 'fr:angle', 'fr:angleterre', 'caribbean netherlands', 'south sudan', 'нидерланды', 'antarctic', '日本', 'schweiz', 'fr:international', 'east germany', 'es:santona', 'turks and caicos islands', 'palau', 'central african republic

In [46]:
# 1. Zählen und sortieren (Pandas macht das bei value_counts standardmäßig)
counts = df["countries_en"].str.split(",").explode().str.strip().value_counts()

# 2. & 3. Kumulierte Prozente berechnen
cum_percentage = counts.cumsum() / counts.sum()

# 4. Den Filter anwenden (Alle Länder, deren Wert <= 0.90 ist)
top_90_percent_countries = cum_percentage[cum_percentage <= 0.90]

# Das Ergebnis anzeigen
print(top_90_percent_countries)

# Den Haupt-Datensatz filtern:
df_final = df[df["countries_en"].isin(top_90_percent_countries.index)]

countries_en
france            0.264485
united states     0.454259
germany           0.539621
spain             0.614018
italy             0.669962
united kingdom    0.708900
canada            0.734069
switzerland       0.755766
belgium           0.776856
ireland           0.793377
netherlands       0.809153
world             0.824708
australia         0.839989
japan             0.848099
romania           0.855647
poland            0.862837
russia            0.870009
brazil            0.877124
norway            0.882448
sweden            0.887654
morocco           0.892372
austria           0.896868
Name: count, dtype: float64


In [47]:
top_50_laender = top_x_spalte(df, "countries_en", 50)
top_50_laender

countries_en
france                  1256554
united states            901601
germany                  405550
spain                    353457
italy                    265788
united kingdom           184990
canada                   119574
switzerland              103082
belgium                  100200
ireland                   78491
netherlands               74949
world                     73899
australia                 72601
japan                     38531
romania                   35857
poland                    34161
russia                    34075
brazil                    33802
norway                    25294
sweden                    24732
morocco                   22417
austria                   21359
portugal                  21025
india                     20867
finland                   19035
czech republic            18880
denmark                   18455
mexico                    16765
bulgaria                  16329
argentina                 15090
saudi arabia              1

In [48]:
bottom_10_laender = bottom_x_spalte(df, "countries_en", 10)
bottom_10_laender

countries_en
it:ringo-gelato-cacao    1
suede                    1
Australia                1
belgio                   1
germania                 1
svizzera                 1
polonia                  1
es:asturias              1
saba                     1
snacks                   1
Name: count, dtype: int64

In [49]:
labels_spalten = ["labels", "labels_tags", "labels_en"]
all_labels = df[labels_spalten].sample(20)
all_labels

,labels,labels_tags,labels_en
92390,NaN,NaN,NaN
4360935,NaN,NaN,NaN
4314572,NaN,NaN,NaN
904563,NaN,NaN,NaN
3908161,NaN,NaN,NaN
4023518,NaN,NaN,NaN
837411,NaN,NaN,NaN
3347110,NaN,NaN,NaN
4140642,NaN,NaN,NaN
2928451,NaN,NaN,NaN


In [50]:
#Prozent und Anzahl der Spalte vor Bearbeitung
labels_en_completeness = str(df["labels_en"].notna().mean()*100)
labels_en_before = df["labels_en"].notna().sum()

print("Vor Zusammenführen war Spalte zu " + labels_en_completeness + " gefüllt")

#labels_en mit labels_tags auffüllen falls leer, mit labels auffüllen falls auch leer
df["labels_en"] = (df["labels_en"]
                                .fillna(df["labels_tags"]
                                .fillna(df["labels"])))

#Prozent und Anzahl der Spalte nach Bearbeitung
labels_en_completeness = str(df["labels_en"].notna().mean()*100)
labels_en_after = df["labels_en"].notna().sum()

labels_added = str(labels_en_after - labels_en_before)

print("Nach Zusammenführen ist Spalte zu " + labels_en_completeness + " gefüllt")
print("Es wurden " + labels_added + " Einträge hinzugefügt")

# labels und labels_tags löschen
df = df.drop(columns=["labels", "labels_tags"])

Vor Zusammenführen war Spalte zu 27.225015623323394 gefüllt
Nach Zusammenführen ist Spalte zu 27.225859827075972 gefüllt
Es wurden 38 Einträge hinzugefügt


In [51]:
'''
Erstelle ein Wörterbuch, um die folgenden Labels zu übersetzen

'''

labels_dict = {
    'EG-Öko-Verordnung': 'eu organic',
    'Ohne Gentechnik': 'no gmos',
    'fr:eco-emballages': 'green dot',
    'pt:ecoponto-amarelo': 'yellow ecopoint',
    'Sistema de Etiquetado Frontal de Alimentos y Bebidas': 'front-of-pack nutrition labelling',
    'fr:Origine France': 'made in france', # to 'french origin'
    'fr:Sélection Intermarché': 'intermarché selection',
    'fr:Bleu Blanc Cœur': 'bleu blanc cœur'
}

#zu_loeschen = r'\bNew\b'
#df = delete_rows_with_words_in_spalte(df, "labels_en", zu_loeschen)

df['labels_en'] = df['labels_en'].apply(lambda x: uebersetze_mit_dict(x, labels_dict))

In [52]:
# 1. Zählen und sortieren (Pandas macht das bei value_counts standardmäßig)
counts = df["labels_en"].str.split(",").explode().str.strip().value_counts()

# 2. & 3. Kumulierte Prozente berechnen
cum_percentage = counts.cumsum() / counts.sum()

# 4. Den Filter anwenden (Alle Labels, deren Wert <= 0.90 ist)
top_90_percent_labels = cum_percentage[cum_percentage <= 0.90]

# Das Ergebnis anzeigen
print(top_90_percent_labels)

labels_en
organic                                                 0.088988
no gluten                                               0.160230
eu organic                                              0.220483
vegetarian                                              0.273474
green dot                                               0.322070
vegan                                                   0.367784
nutriscore                                              0.408603
ab agriculture biologique                               0.434557
no gmos                                                 0.460508
no preservatives                                        0.486073
non gmo project                                         0.508205
made in france                                          0.524496
no colorings                                            0.540028
no added sugar                                          0.554116
no lactose                                              0.566911
eu agriculture 

In [53]:
#Prozent und Anzahl der Spalte vor Bearbeitung
brands_en_completeness = str(df["brands_en"].notna().mean()*100)
brands_en_before = df["brands_en"].notna().sum()

print("Vor Zusammenführen war Spalte zu " + brands_en_completeness + " gefüllt")

#brands_en behalten, mit brands_tags auffüllen falls leer, mit brands auffüllen falls auch leer
df["brands_en"] = df["brands_en"].fillna(df["brands_tags"].fillna(df["brands"]))

#Prozent und Anzahl der Spalte nach Bearbeitung
brands_en_completeness = str(df["brands_en"].notna().mean()*100)
brands_en_after = df["brands_en"].notna().sum()

brands_en_added = str(brands_en_after - brands_en_before)

print("Nach Zusammenführen ist Spalte zu " + brands_en_completeness + " gefüllt")
print("Es wurden " + brands_en_added + " Einträge hinzugefügt")

df = df.drop(columns=["brands", "brands_tags"])

Vor Zusammenführen war Spalte zu 62.726160519123106 gefüllt
Nach Zusammenführen ist Spalte zu 62.744044309144755 gefüllt
Es wurden 805 Einträge hinzugefügt


In [54]:
top_90_brands = top_x_percent_spalte(df, "brands_en", 0.90)
print(top_90_brands)

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [55]:
# Erstellte eine neue Spalte "brands_clean", die die bereinigten Markennamen enthält
df = clean_brands_pandas(df, 1000)
brands_clean = df["brands_clean"]
brands_clean.sample(20)

/tmp/ipykernel_33032/1968996030.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['brands_clean'] = df['brands_en'].str.lower().str.strip()


4049760        other
253317         other
3531500        other
3031199        other
3439960        other
287845         other
1068188        other
4489868        other
2138274        other
4370428        other
726240     hacendado
2395093        other
4240224        other
3904013        other
1394530        other
611173         other
2161694        other
771506         other
3831288        other
3107893        other
Name: brands_clean, dtype: str

In [56]:
df['brands_en'] = df['brands_en'].str.lower().str.strip()
    
# 2. Akzente entfernen (é -> e, ä -> a)
# Wichtig: Benötigt oft das 'unidecode' Paket oder Pandas string normalize
df['brands_en'] = df['brands_en'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')
    
# 3. Satzzeichen und Firmenzusätze (GmbH, Inc, SA) mit Regex entfernen
# Ersetzt alles was kein Buchstabe oder Zahl ist mit einem Leerzeichen
df['brands_en'] = df['brands_en'].str.replace(r'[^a-z0-9\s]', ' ', regex=True)
# Entfernt gängige Rechtsformen
df['brands_en'] = df['brands_en'].str.replace(r'\b(gmbh|inc|ltd|sa|co|kg|ag)\b', '', regex=True)
    
# Letzte doppelte Leerzeichen entfernen, die durch das Löschen entstanden sind
df['brands_en'] = df['brands_en'].str.replace(r'\s+', ' ', regex=True).str.strip()

In [57]:
origins_spalten = ["origins", "origins_tags", "origins_en"]
origins = df[origins_spalten]
origins.sample(20)

,origins,origins_tags,origins_en
2567627,NaN,NaN,NaN
1363733,NaN,NaN,NaN
2063381,NaN,NaN,NaN
1281407,NaN,NaN,NaN
2702222,NaN,NaN,NaN
3612821,NaN,NaN,NaN
3944766,NaN,NaN,NaN
4060224,NaN,NaN,NaN
2226020,NaN,NaN,NaN
90345,NaN,NaN,NaN


In [58]:
contains_california = contains_x_in_spalte(df, "origins_en", "fr:porc-origine-france")
contains_california.sample(20)

,code,url,creator,created_t,created_datetime,last_modified_t,last_modified_datetime,last_modified_by,last_updated_t,last_updated_datetime,product_name,abbreviated_product_name,generic_name,quantity,packaging,packaging_tags,packaging_en,packaging_text,brands_en,categories,categories_tags,categories_en,origins,origins_tags,origins_en,manufacturing_places,manufacturing_places_tags,labels_en,emb_codes,emb_codes_tags,first_packaging_code_geo,cities,cities_tags,purchase_places,stores,countries_en,ingredients_text,ingredients_tags,ingredients_analysis_tags,allergens,traces,traces_tags,traces_en,serving_size,serving_quantity,no_nutrition_data,additives_n,additives,additives_tags,additives_en,nutriscore_score,nutriscore_grade,nova_group,pnns_groups_1,pnns_groups_2,food_groups,food_groups_tags,food_groups_en,states_en,brand_owner,environmental_score_score,environmental_score_grade,nutrient_levels_tags,product_quantity,owner,data_quality_errors_tags,unique_scans_n,popularity_tags,completeness,last_image_t,last_image_datetime,main_category,main_category_en,image_url,image_small_url,image_ingredients_url,image_ingredients_small_url,image_nutrition_url,image_nutrition_small_url,energy-kcal_100g,energy_100g,fat_100g,saturated-fat_100g,butyric-acid_100g,caproic-acid_100g,caprylic-acid_100g,capric-acid_100g,lauric-acid_100g,myristic-acid_100g,palmitic-acid_100g,stearic-acid_100g,arachidic-acid_100g,behenic-acid_100g,lignoceric-acid_100g,cerotic-acid_100g,montanic-acid_100g,melissic-acid_100g,unsaturated-fat_100g,monounsaturated-fat_100g,omega-9-fat_100g,polyunsaturated-fat_100g,omega-3-fat_100g,omega-6-fat_100g,alpha-linolenic-acid_100g,eicosapentaenoic-acid_100g,docosahexaenoic-acid_100g,linoleic-acid_100g,arachidonic-acid_100g,gamma-linolenic-acid_100g,dihomo-gamma-linolenic-acid_100g,oleic-acid_100g,elaidic-acid_100g,gondoic-acid_100g,mead-acid_100g,erucic-acid_100g,nervonic-acid_100g,trans-fat_100g,cholesterol_100g,carbohydrates_100g,sugars_100g,added-sugars_100g,sucrose_100g,glucose_100g,fructose_100g,galactose_100g,lactose_100g,maltose_100g,maltodextrins_100g,psicose_100g,starch_100g,polyols_100g,erythritol_100g,isomalt_100g,maltitol_100g,sorbitol_100g,fiber_100g,soluble-fiber_100g,polydextrose_100g,insoluble-fiber_100g,proteins_100g,casein_100g,serum-proteins_100g,nucleotides_100g,salt_100g,added-salt_100g,sodium_100g,alcohol_100g,vitamin-a_100g,beta-carotene_100g,vitamin-d_100g,vitamin-e_100g,vitamin-k_100g,vitamin-c_100g,vitamin-b1_100g,vitamin-b2_100g,vitamin-pp_100g,vitamin-b6_100g,vitamin-b9_100g,folates_100g,vitamin-b12_100g,biotin_100g,pantothenic-acid_100g,silica_100g,bicarbonate_100g,potassium_100g,chloride_100g,calcium_100g,phosphorus_100g,iron_100g,magnesium_100g,zinc_100g,copper_100g,manganese_100g,fluoride_100g,selenium_100g,chromium_100g,molybdenum_100g,iodine_100g,caffeine_100g,taurine_100g,methylsulfonylmethane_100g,ph_100g,fruits-vegetables-legumes_100g,collagen-meat-protein-ratio_100g,cocoa_100g,chlorophyl_100g,carbon-footprint_100g,glycemic-index_100g,water-hardness_100g,choline_100g,phylloquinone_100g,beta-glucan_100g,inositol_100g,carnitine_100g,sulphate_100g,nitrate_100g,acidity_100g,carbohydrates-total_100g,water_100g,ash_100g,brands_clean
2056075,3660140003841,http://world-en.openfoodfacts.org/product/3660...,kiliweb,1516383210,2018-01-19T17:33:30Z,1774276787,2026-03-23T14:39:47Z,new-nutrition-bot,1.774277e+09,2026-03-23T14:39:47Z,La pizza duo de manosque,NaN,NaN,570 g e,"Carton, Film plastique","en:cardboard,fr:film-plastique","Cardboard,fr:film-plastique",NaN,manosque,NaN,NaN,NaN,fr:Porc origine France,fr:porc-origine-france,fr:porc-origine-france,"Provence,France","provence,france","french meat, french pork, fr:fabrique-en-provence",emb 04112e,emb-04112e,"43.833333,5.783333",NaN,manosque-alpes-de-haute-provence-france,France,Carrefour,france,"Garniture (51%) : purée de tomate de Provence,...","en:filling,en:emmental,en:dairy,en:cheese,en:c...","en:palm-oil-free,en:non-vegan,en:non-vegetarian","gluten,milk","en

In [59]:
top_90_origins = top_x_percent_spalte(df, "origins_en", 0.90)
print(top_90_origins)

to_fix = ['unspecified', 'unknown', 'european union and non european union', 'non european union', 'great britain',
          'north-east atlantic ocean', 'fr:lait-origine-france', 'fr:import', "provence-alpes-côte d'azur", 'provence',
          'fr:ecosse', "côte d'ivoire", 'nouvelle-aquitaine', 'california', 'south-western france', 'fr:hors-france',
          'fr:porc-origine-france', 'rhône-alpes', 'auvergne-rhône-alpes',  'alsace', 'es:agricultura-ue', 'world',
          'es:agricultura-no-ue', 'fr:hors-union-europeenne', 'south-america', 'camargue', 'fr:sud-ouest',
          'fr:agriculture-ue-non-ue', 'pays de la loire', 'québec', 'imported-unknown', 'auvergne', 'brittany', 'normandy',
          'northeast pacific ocean', 'bayern', 'northwest pacific ocean']

to_fix_to_80 = []

origins_en
france                                                                0.199082
italy                                                                 0.259166
european union                                                        0.301389
spain                                                                 0.342411
germany                                                               0.382519
united states                                                         0.403870
mexico                                                                0.422781
united kingdom                                                        0.440604
brazil                                                                0.453374
argentina                                                             0.465942
switzerland                                                           0.478300
canada                                                                0.489376
poland                                   

In [60]:
#Prozent und Anzahl der Spalte vor Bearbeitung
origins_en_completeness = str(df["origins_en"].notna().mean()*100)
origins_en_before = df["origins_en"].notna().sum()

print("Vor Zusammenführen war Spalte zu " + origins_en_completeness + " gefüllt")

#origins_en behalten, mit origins_tags auffüllen falls leer, mit origins auffüllen falls auch leer
df["origins_en"] = df["origins_en"].fillna(df["origins_tags"].fillna(df["origins"]))

#Prozent und Anzahl der Spalte nach Bearbeitung
origins_en_completeness = str(df["origins_en"].notna().mean()*100)
origins_en_after = df["origins_en"].notna().sum()

origins_en_added = str(origins_en_after - origins_en_before)

print("Nach Zusammenführen ist Spalte zu " + origins_en_completeness + " gefüllt")
print("Es wurden " + origins_en_added + " Einträge hinzugefügt")

df = df.drop(columns=["origins", "origins_tags"])

Vor Zusammenführen war Spalte zu 3.8319963441534335 gefüllt
Nach Zusammenführen ist Spalte zu 3.834928841399219 gefüllt
Es wurden 132 Einträge hinzugefügt


In [61]:
origins_mapping = {
    # --- Ursprüngliche Einträge ---
    'unspecified': None, 
    'unknown': None, 
    'european union and non european union': None, 
    'non european union': None, 
    'fr:import': None, 
    'fr:hors-france': None,
    'great britain': 'united kingdom', 
    'fr:ecosse': 'united kingdom',
    'fr:lait-origine-france': 'france', 
    'fr:porc-origine-france': 'france',
    "provence-alpes-côte d'azur": 'france', 
    'provence': 'france',
    'nouvelle-aquitaine': 'france', 
    'south-western france': 'france',
    'rhône-alpes': 'france', 
    'auvergne-rhône-alpes': 'france', 
    'alsace': 'france',
    'california': 'united states', 
    "côte d'ivoire": 'ivory coast',
    'es:agricultura-ue': 'european union', 
    'north-east atlantic ocean': 'atlantic ocean',
    
    # --- Vorherige Erweiterung ---
    'world': None,                                 
    'es:agricultura-no-ue': None,                  
    'fr:hors-union-europeenne': None,              
    'south-america': 'south america',              
    'camargue': 'france',                          
    'fr:sud-ouest': 'france',                      
    'fr:agriculture-ue-non-ue': None,              
    'pays de la loire': 'france',                  
    'québec': 'canada',                            
    'imported-unknown': None,                      
    'auvergne': 'france',
    
    # --- NEUESTE Einträge ---
    'brittany': 'france',                          # Region in Frankreich (Bretagne)
    'normandy': 'france',                          # Region in Frankreich (Normandie)
    'bayern': 'germany',                           # Bundesland in Deutschland
    'northeast pacific ocean': 'pacific ocean',    # Ozean
    'northwest pacific ocean': 'pacific ocean'     # Ozean
}

df["origins_en"] = df["origins_en"].apply(lambda x: uebersetze_mit_dict(x, origins_mapping))

In [62]:
pnns_groups_1_completeness = df["pnns_groups_1"].notna().mean()*100
print(pnns_groups_1_completeness)

food_groups_1_completeness = df["food_groups_en"].notna().mean()*100
print(food_groups_1_completeness)

99.00628331966685
32.207150716806744


In [63]:
def find_inconsistent_groups(df):
    # --- NEU: Harmonisierung bekannter OpenFoodFacts Tippfehler/Mappings ---
    # Wir passen die englische Spalte temporär an (z.B. Plural zu Singular), 
    # nutzen sie aber NUR für die Textvergleiche unten.
    food_groups_clean = df['food_groups_en'].astype(str)\
        .str.replace('Fats and sauces', 'Fat and sauces', regex=False)
        # .str.replace('Anderer Fehler', 'Richtiger Wert', regex=False) # <- Hier kannst du später weitere anketten

    # Fall 1: Echte NaNs (Hier nutzen wir die Originalspalte, da .astype(str) aus NaN "nan" macht)
    mask_nan = df['food_groups_en'].isna() & \
               (df['pnns_groups_1'] == 'unknown') & \
               (df['pnns_groups_2'] == 'unknown')

    # Fall 2: Beide PNNS Spalten sind gleich (Abgleich mit der harmonisierten Spalte)
    mask_duplicate = (df['pnns_groups_1'] == df['pnns_groups_2']) & \
                     (food_groups_clean == df['pnns_groups_1'].astype(str))

    # Fall 3: Der Standardfall -> "PNNS_1,PNNS_2" (Abgleich mit der harmonisierten Spalte)
    mask_concat = (food_groups_clean == df['pnns_groups_1'].astype(str) + ',' + df['pnns_groups_2'].astype(str))

    # Fall 4: Toleranter Abgleich (ohne Leerzeichen und Kommas)
    clean_food_str = food_groups_clean.str.replace(', ', '', regex=False).str.replace(',', '', regex=False).str.replace(' ', '', regex=False)
    clean_pnns_str = (df['pnns_groups_1'].astype(str) + df['pnns_groups_2'].astype(str)).str.replace(' ', '', regex=False)
    mask_fuzzy = clean_food_str == clean_pnns_str

    # Fall 5: Alle drei Spalten sind NaN
    mask_all_nan = df['food_groups_en'].isna() & \
                   df['pnns_groups_1'].isna() & \
                   df['pnns_groups_2'].isna()

    # Wir kombinieren alle akzeptierten Muster
    is_consistent = mask_nan | mask_duplicate | mask_concat | mask_fuzzy | mask_all_nan

    # Gibt alle Zeilen zurück, die immer noch abweichen
    return df[~is_consistent]

# Anwendung auf deinen DataFrame:
inconsistent_rows = find_inconsistent_groups(df)

# Ausgabe der Problemfälle, um neue Muster zu entdecken
print(f"Gefundene Abweichungen: {len(inconsistent_rows)}")
print(inconsistent_rows[['food_groups_en', 'pnns_groups_1', 'pnns_groups_2']].head(10))

Gefundene Abweichungen: 245016
                                    food_groups_en   pnns_groups_1       pnns_groups_2
322                               Fish‚ Meat‚ Eggs         unknown             unknown
355  Fish‚ Meat‚ Eggs,Meat,Meat other than poultry  Fish Meat Eggs                Meat
418              Fish‚ Meat‚ Eggs,Fish and seafood  Fish Meat Eggs    Fish and seafood
441                                            NaN   Sugary snacks              Sweets
467              Fish‚ Meat‚ Eggs,Fish and seafood  Fish Meat Eggs    Fish and seafood
518   Fish‚ Meat‚ Eggs,Fish and seafood,Fatty fish   Sugary snacks  Biscuits and cakes
528  Fish‚ Meat‚ Eggs,Meat,Meat other than poultry  Fish Meat Eggs                Meat
535                                            NaN       Beverages        Fruit juices
566                Fish‚ Meat‚ Eggs,Processed meat  Fish Meat Eggs      Processed meat
593  Fish‚ Meat‚ Eggs,Meat,Meat other than poultry  Fish Meat Eggs                Meat


In [64]:
print(inconsistent_rows[['food_groups_en', 'pnns_groups_1', 'pnns_groups_2']].sample(10))

                                        food_groups_en   pnns_groups_1     pnns_groups_2
1776652   Fish‚ Meat‚ Eggs,Fish and seafood,Fatty fish  Fish Meat Eggs  Fish and seafood
3855480                          Fish‚ Meat‚ Eggs,Eggs  Fish Meat Eggs              Eggs
4072267                Fish‚ Meat‚ Eggs,Processed meat  Fish Meat Eggs    Processed meat
2194478                Baby foods and milks,Baby foods      Baby foods        Baby foods
1876528   Fish‚ Meat‚ Eggs,Fish and seafood,Fatty fish  Fish Meat Eggs  Fish and seafood
1841217                Fish‚ Meat‚ Eggs,Processed meat  Fish Meat Eggs    Processed meat
1419807              Fish‚ Meat‚ Eggs,Fish and seafood  Fish Meat Eggs  Fish and seafood
386134                           Fruits and vegetables         unknown           unknown
1480082                        Fish‚ Meat‚ Eggs,Offals  Fish Meat Eggs            Offals
2416881  Fish‚ Meat‚ Eggs,Meat,Meat other than poultry  Fish Meat Eggs              Meat


In [65]:
essen_spalten = ["food_groups_en", "pnns_groups_1", "pnns_groups_2"]
essen = df[essen_spalten]
essen.sample(20)

,food_groups_en,pnns_groups_1,pnns_groups_2
884062,NaN,unknown,unknown
917359,NaN,unknown,unknown
994278,NaN,unknown,unknown
1831681,"Fish‚ Meat‚ Eggs,Fish and seafood",Fish Meat Eggs,Fish and seafood
3076706,"Sugary snacks,Sweets",Sugary snacks,Sweets
1167918,NaN,unknown,unknown
708649,NaN,unknown,unknown
4221826,NaN,unknown,unknown
2198588,"Fats and sauces,Dressings and sauces",Fat and sauces,Dressings and sauces
2700744,NaN,unknown,unknown


In [66]:
def fill_missing_pnns_groups(df):
    # 1. Maske definieren (Die erste, striktere Variante)
    mask_pnns_1 = df['pnns_groups_1'].isna() | (df['pnns_groups_1'] == 'unknown')
    mask_pnns_2 = df['pnns_groups_2'].isna() | (df['pnns_groups_2'] == 'unknown')
    mask_food = df['food_groups_en'].notna() & (df['food_groups_en'] != 'NaN')
    
    trigger_condition = mask_pnns_1 & mask_pnns_2 & mask_food

    # --- NEU: Zählen und Ausgeben der betroffenen Zeilen ---
    num_affected_rows = trigger_condition.sum()
    print(f"Info: {num_affected_rows} Zeilen wurden erfolgreich aufgefüllt.")

    # Falls keine Zeile zutrifft, direkt den DataFrame zurückgeben
    if not trigger_condition.any():
        return df

    # 2. Nur die betroffenen Zeilen extrahieren
    subset = df.loc[trigger_condition, 'food_groups_en'].astype(str)

    # 3. Am LETZTEN Komma aufteilen
    split_df = subset.str.rsplit(',', n=1, expand=True)

    if split_df.shape[1] == 1:
        split_df[1] = split_df[0]
    else:
        split_df[1] = split_df[1].fillna(split_df[0])

    # 4. Bereinigung nach deinem Beispiel
    pnns_1_clean = split_df[0].str.replace(',', '', regex=False).str.strip()
    pnns_2_clean = split_df[1].str.strip()

    # 5. Die bereinigten Werte zurückschreiben
    df.loc[trigger_condition, 'pnns_groups_1'] = pnns_1_clean
    df.loc[trigger_condition, 'pnns_groups_2'] = pnns_2_clean

    return df

# Anwendung auf deinen DataFrame:
df = fill_missing_pnns_groups(df)

Info: 9120 Zeilen wurden erfolgreich aufgefüllt.


In [67]:
df = df.drop(columns=["food_groups_en", "food_groups", "food_groups_tags"])

In [68]:
top_90_pnns_groups_1 = top_x_percent_spalte(df, "pnns_groups_1", 1)
top_90_pnns_groups_1

pnns_groups_1
unknown                    0.674687
sugary snacks              0.740107
fish meat eggs             0.783730
cereals and potatoes       0.825467
milk and dairy products    0.864826
beverages                  0.898726
fat and sauces             0.926870
fruits and vegetables      0.950294
composite foods            0.973554
salty snacks               0.990423
alcoholic beverages        0.998821
baby foods                 0.999863
fish‚ meat‚ eggs           1.000000
590                        1.000000
Name: count, dtype: float64

In [69]:
to_fix_pnns = ['590', 'fish, meat, eggs']
pnns_1_dict = {
    'fish, meat, eggs': 'fish meat eggs'
}

df["pnns_groups_1"] = df["pnns_groups_1"].apply(lambda x: uebersetze_mit_dict(x, pnns_1_dict))

In [70]:
top_90_pnns_2 = top_x_percent_spalte(df, "pnns_groups_2", 1)
top_90_pnns_2

pnns_groups_2
unknown                             0.674687
sweets                              0.704123
biscuits and cakes                  0.729999
dressings and sauces                0.749738
cereals                             0.767868
cheese                              0.785907
one-dish meals                      0.801892
processed meat                      0.817409
milk and yogurt                     0.830949
meat                                0.844331
fish and seafood                    0.855898
vegetables                          0.867301
bread                               0.878032
sweetened beverages                 0.886731
fats                                0.895137
alcoholic beverages                 0.903536
appetizers                          0.910958
fruits                              0.917972
chocolate products                  0.924932
unsweetened beverages               0.930836
breakfast cereals                   0.936560
salty and fatty products            0.941

In [71]:
pnns_2_dict = {
    "fresse": "undefined"
}

df["pnns_groups_2"] = df["pnns_groups_2"].apply(lambda x: uebersetze_mit_dict(x, pnns_2_dict))

In [72]:
#Prozent und Anzahl der Spalte vor Bearbeitung
main_category_en_completeness = str(df["main_category_en"].notna().mean()*100)
main_category_en_before = df["main_category_en"].notna().sum()

print("Vor Zusammenführen war Spalte zu " + main_category_en_completeness + " gefüllt")

#main_category_en behalten, mit main_category auffüllen falls leer
df["main_category_en"] = df["main_category_en"].fillna(df["main_category"])

#Prozent und Anzahl der Spalte nach Bearbeitung
main_category_en_completeness = str(df["main_category_en"].notna().mean()*100)
main_category_en_after = df["main_category_en"].notna().sum()

main_category_en_added = str(main_category_en_after - main_category_en_before)

print("Nach Zusammenführen ist Spalte zu " + main_category_en_completeness + " gefüllt")
print("Es wurden " + main_category_en_added + " Einträge hinzugefügt")

df = df.drop(columns=["main_category"])

Vor Zusammenführen war Spalte zu 41.08461965177484 gefüllt
Nach Zusammenführen ist Spalte zu 41.08466408355129 gefüllt
Es wurden 2 Einträge hinzugefügt


In [73]:
top_90_main_category = top_x_percent_spalte(df, "main_category_en", 0.90)
top_90_main_category

main_category_en
groceries                                                                                                     0.034790
beverages                                                                                                     0.055086
snacks                                                                                                        0.074302
undefined                                                                                                     0.093248
biscuits                                                                                                      0.106555
cheeses                                                                                                       0.118164
confectioneries                                                                                               0.128567
breads                                                                                                        0.137114
yogurts                        

In [74]:
def clean_main_category_pandas(df, top_n=200):
    # 1. Arbeite auf einer Kopie der Spalte oder überschreibe sie
    df_cat = df.copy()
    
    # 2. Präfixe (wie 'en:', 'fr:') am Anfang des Strings entfernen
    # ^[a-z]{2,3}: sucht nach 2-3 Buchstaben am Anfang (^), gefolgt von einem Doppelpunkt
    df["main_category_en"] = df['main_category_en'].str.replace(r'^en:', '', regex=True)
    df_cat['main_category_clean'] = df_cat['main_category_en'].str.replace(r'^en:', '', regex=True)
    
    # 3. Bindestriche durch Leerzeichen ersetzen, trimmen und in Kleinbuchstaben umwandeln
    df_cat['main_category_en'] = df['main_category_en'].str.replace('-', ' ').str.strip().str.lower()
    df_cat['main_category_clean'] = df_cat['main_category_clean'].str.replace('-', ' ').str.strip().str.lower()
    
    # 4. Thresholding: Den "Long Tail" in 'other' zusammenfassen
    # Behalte nur die Top N (z.B. Top 100) Kategorien, der Rest wird 'other'
    top_categories = df_cat['main_category_clean'].value_counts().nlargest(top_n).index
    df_cat.loc[~df_cat['main_category_clean'].isin(top_categories), 'main_category_clean'] = 'other'
    
    return df_cat

df = clean_main_category_pandas(df, 200)

In [75]:
purchase_places_completeness = df["purchase_places"].notna().mean()*100
purchase_places_completeness

np.float64(4.671290385430109)

In [76]:
top_90_purchase_places = top_x_percent_spalte(df, "purchase_places", 0.9)
top_90_purchase_places


purchase_places
france                           0.280190
deutschland                      0.332539
españa                           0.356655
portugal                         0.378166
canada                           0.399377
united kingdom                   0.418393
paris                            0.434320
alberta                          0.448133
calgary                          0.461875
lyon                             0.474195
italia                           0.486291
nantes                           0.497203
united states                    0.507847
courrières                       0.517201
belgique                         0.526356
suisse                           0.535276
česká republika                  0.543873
polska                           0.551772
norway                           0.559287
česko                            0.566691
czech republic                   0.573981
madrid                           0.581176
bolivia                          0.588212
sweden            

In [77]:
to_fix_purchase = ["deutschland", "españa", "italia", "belgique", "suisse", "česká republika", "polska", "česko",
                   "usa", "méxico", "sverige", "normandie", "bayern", "praha", "tunisie", "münster", "nederland",
                   "österreich", "uk", "россия"]

purchase_places_mapping = {
    "deutschland": "germany",
    "españa": "spain",
    "italia": "italy",
    "belgique": "belgium",
    "suisse": "switzerland",
    "česká republika": "czech republic",
    "polska": "poland",
    "česko": "czech republic",
    "usa": "united states",
    "méxico": "mexico",
    "sverige": "sweden",
    
    # Regionen und Städte beibehalten (auf Englisch übersetzt)
    "normandie": "normandy",     # Region in Frankreich
    "bayern": "bavaria",         # Bundesland in Deutschland
    "praha": "prague",           # Stadt (Prag)
    "münster": "munster",        # Stadt in Deutschland
    
    "tunisie": "tunisia",
    "nederland": "netherlands",
    "österreich": "austria",
    "uk": "united kingdom",
    "россия": "russia"           # Russisch für Russland
}

df["purchase_places"] = df["purchase_places"].apply(lambda x: uebersetze_mit_dict(x, purchase_places_mapping))

In [78]:
brand_owner_completeness = df["brand_owner"].notna().mean()*100
brand_owner_completeness

np.float64(7.05185610413742)

In [79]:
top_90_brand_owner = top_x_percent_spalte(df, "brand_owner", 0.9)
top_90_brand_owner

brand_owner
inc.                                                                                    0.211310
llc                                                                                     0.240581
wal-mart stores                                                                         0.253374
target stores                                                                           0.264508
topco associates                                                                        0.275341
meijer                                                                                  0.285433
safeway                                                                                 0.294937
the kroger co.                                                                          0.304228
hy-vee                                                                                  0.312483
ahold usa                                                                               0.320345
supervalu         

In [80]:
brands_dict = {'dummy_value' : 'strip und lower'}

df["brand_owner"] = df["brand_owner"].apply(lambda x: uebersetze_mit_dict(x, brands_dict))

In [81]:
popularity_tags_completeness = df["popularity_tags"].notna().mean()*100
popularity_tags_completeness

np.float64(34.96192085678683)

In [82]:
top_90_popularity = top_x_percent_spalte(df, "popularity_tags", 0.9)
top_90_popularity

popularity_tags
top-75-percent-scans-2025       0.021819
top-80-percent-scans-2025       0.043637
top-85-percent-scans-2025       0.065456
top-90-percent-scans-2025       0.087275
top-75-percent-scans-2024       0.106744
top-80-percent-scans-2024       0.126214
top-85-percent-scans-2024       0.145683
top-90-percent-scans-2024       0.165153
top-75-percent-scans-2023       0.180901
top-80-percent-scans-2023       0.196649
top-85-percent-scans-2023       0.212396
top-90-percent-scans-2023       0.228144
top-85-percent-scans-2021       0.241679
top-90-percent-scans-2021       0.255214
top-90-percent-scans-2020       0.268039
top-85-percent-scans-2020       0.280118
top-90-percent-scans-2019       0.291406
top-90-percent-scans-2022       0.302300
top-85-percent-scans-2022       0.312699
top-80-percent-scans-2021       0.322113
top-85-percent-scans-2019       0.330817
bottom-25-percent-scans-2019    0.339144
bottom-25-percent-scans-2020    0.347389
top-80-percent-scans-2022       0.355529


In [83]:
df["popularity_tags"] = df["popularity_tags"].str.replace('-', ' ').str.strip().str.lower()

In [84]:
no_nutrition_completeness = df["no_nutrition_data"].notna().mean()*100
no_nutrition_completeness

np.float64(0.2625029352742318)

In [85]:
top_90_no_nutrition = top_x_percent_spalte(df, "no_nutrition_data", 1)
top_90_no_nutrition

no_nutrition_data
on                                                                                       0.430095
off                                                                                      0.835562
true                                                                                     0.994076
false                                                                                    0.999915
https://images.openfoodfacts.org/images/products/356/007/112/6346/front_fr.62.400.jpg    1.000000
Name: count, dtype: float64

In [86]:
df["no_nutrition_data"] = df["no_nutrition_data"].str.strip().str.lower()

In [87]:
cities_tags_completeness = completeness_spalte(df, "cities_tags")
cities_tags_completeness

np.float64(2.2627548634467103)

In [88]:
top_90_cities_tags = top_x_percent_spalte(df, "cities_tags", 0.9)
top_90_cities_tags

cities_tags
sable-sur-sarthe-sarthe-france                             0.016204
villers-bocage-calvados-france                             0.024803
bannalec-finistere-france                                  0.033245
lamballe-cotes-d-armor-france                              0.040021
theix-morbihan-france                                      0.046215
kervignac-morbihan-france                                  0.052368
saint-evarzec-finistere-france                             0.058488
saint-martin-des-entrees-calvados-france                   0.064559
saint-denis-de-l-hotel-loiret-france                       0.070281
avignon-vaucluse-france                                    0.075679
chateaubourg-ille-et-vilaine-france                        0.081078
corbas-rhone-france                                        0.086460
douarnenez-finistere-france                                0.091767
boulogne-sur-mer-pas-de-calais-france                      0.096983
ars-sur-moselle-moselle-france      

In [89]:
df["cities_tags"] = df["cities_tags"].str.replace("-", " ").str.strip().str.lower()

In [90]:
#Meme Bereinigung

In [91]:
section_header("Teil A  ·  product_name · packaging · additives · ingredients · manufacturing")

# Packaging — REINHEIT: nur strukturierte Quellen, Praefixe -> lesbar (entfernt). Wie Hauptnotebook.
packaging_before = df["packaging_en"].notna().sum()
df["packaging_en"] = clean_series(df["packaging_en"].astype(str).where(df["packaging_en"].notna(), np.nan))
df["packaging_en"] = df["packaging_en"].fillna(normalize_tags(df["packaging_tags"]))
merge_report("packaging_en", packaging_before, df["packaging_en"].notna().sum())
df = df.drop(columns=["packaging", "packaging_tags", "packaging_text"])

# Additives — lesbare englische Spalte als Survivor, aus Tags auffuellen, Tags verwerfen
additives_before = df["additives_en"].notna().sum()
df["additives_en"] = clean_series(df["additives_en"].astype(str).where(df["additives_en"].notna(), np.nan))
df["additives_en"] = df["additives_en"].fillna(df["additives_tags"])
merge_report("additives_en", additives_before, df["additives_en"].notna().sum())
df = df.drop(columns=["additives"])  # additives_tags BEHALTEN — wird im NOVA-Teil gebraucht

# Ingredients — Tags sauber & zaehlbar halten; ingredients_text als eigene Spalte ERHALTEN
df["ingredients_tags"] = clean_series(df["ingredients_tags"].astype(str).where(df["ingredients_tags"].notna(), np.nan))
df["ingredients_text"] = df["ingredients_text"].astype("string").str.strip()
print(f"  ingredients_tags sauber gehalten ({df['ingredients_tags'].notna().sum():,})  ·  ingredients_text erhalten ({df['ingredients_text'].notna().sum():,})")

# Manufacturing Places — Rohtext auffuellen, dann Ländernamen -> Englisch übersetzen (Regionen/Städte bleiben)
manufacturing_before = df["manufacturing_places_tags"].notna().sum()
df["manufacturing_places_tags"] = df["manufacturing_places_tags"].fillna(df["manufacturing_places"])
df = df.drop(columns=["manufacturing_places"])
df["manufacturing_places_tags"], _mfg_recognized = clean_manufacturing(df["manufacturing_places_tags"])
_mfg_nn  = int(df["manufacturing_places_tags"].notna().sum())
_mfg_rec = int(_mfg_recognized.sum())
merge_report("manufacturing_places", manufacturing_before, _mfg_nn)
print(f"  {'-> davon einem Land zugeordnet':<26}  [Englisch]   {_mfg_rec / _mfg_nn * 100:5.1f}%   erkannt {_mfg_rec:,} / {_mfg_nn:,}   (Ziel >=80%)")

section_header("Teil B  ·  ingredients_analysis_tags · nova_group · nutrient_levels_tags")

# Ingredients Analysis Tags — kommaseparierte en:-Tags; Junk weg, kleinschreiben, Kommas normalisieren
iat_before = df["ingredients_analysis_tags"].notna().sum()
df["ingredients_analysis_tags"] = (
    clean_series(df["ingredients_analysis_tags"]).str.lower().str.replace(r"\s*,\s*", ",", regex=True)
)
clean_report("ingredients_analysis_tags", iat_before, df["ingredients_analysis_tags"].notna().sum())

# NOVA Group — numerisch erzwingen, nur gueltige Gruppen 1-4 behalten (nullable Int)
nova_before = df["nova_group"].notna().sum()
_nova = pd.to_numeric(df["nova_group"], errors="coerce")
df["nova_group"] = _nova.where(_nova.isin([1, 2, 3, 4])).astype("Int64")
clean_report("nova_group", nova_before, df["nova_group"].notna().sum())

# Nutrient Levels Tags — wie ingredients_analysis_tags
nlt_before = df["nutrient_levels_tags"].notna().sum()
df["nutrient_levels_tags"] = (
    clean_series(df["nutrient_levels_tags"]).str.lower().str.replace(r"\s*,\s*", ",", regex=True)
)
clean_report("nutrient_levels_tags", nlt_before, df["nutrient_levels_tags"].notna().sum())

# ███ 🟢 VERÄNDERT DEN DATENSATZ (df) ███
# TEIL C — NEU: cities_tags · owner · traces · traces_tags · traces_en  (Praefixe BEHALTEN)
section_header("Teil C (neu)  ·  cities_tags · owner · traces · traces_tags · traces_en")

# Tag-Spalten: Junk weg, kleinschreiben, Komma-Abstaende vereinheitlichen — en:/fr: bleiben erhalten
for col in ["cities_tags", "traces", "traces_tags", "traces_en"]:
    before = df[col].notna().sum()
    df[col] = clean_series(df[col]).str.lower().str.replace(r"\s*,\s*", ",", regex=True)
    clean_report(col, before, df[col].notna().sum())

# owner — ID-String: nur saeubern + Whitespace kollabieren; NICHT kleinschreiben, nicht kommasplitten
owner_before = df["owner"].notna().sum()
df["owner"] = clean_series(df["owner"]).str.replace(r"\s+", " ", regex=True).str.strip().replace("", pd.NA)
clean_report("owner", owner_before, df["owner"].notna().sum())


════════════════════════════════════════════════════════════════════
  Teil A  ·  product_name · packaging · additives · ingredients · manufacturing
────────────────────────────────────────────────────────────────────
  packaging_en                [█░░░░░░░░░░░░░░░░░░░]    8.4%   befuellt 378,923  (+1)
  additives_en                [███░░░░░░░░░░░░░░░░░]   15.5%   befuellt 697,070  (+0)
  ingredients_tags sauber gehalten (1,277,639)  ·  ingredients_text erhalten (1,280,953)
  manufacturing_places        [░░░░░░░░░░░░░░░░░░░░]    4.9%   befuellt 219,338  (+5)
  -> davon einem Land zugeordnet  [Englisch]    86.8%   erkannt 190,449 / 219,338   (Ziel >=80%)

════════════════════════════════════════════════════════════════════
  Teil B  ·  ingredients_analysis_tags · nova_group · nutrient_levels_tags
────────────────────────────────────────────────────────────────────
  ingredients_analysis_tags   [██████░░░░░░░░░░░░░░]   30.1%   befuellt 1,356,155  (entfernt 0)
  nova_group               

In [92]:
# ███ 🟢 VERÄNDERT DEN DATENSATZ (df) ███
# ── Spalten umbenennen (nur die gemergten Survivors; cities_tags/traces*/owner behalten ihre Namen)
df = df.rename(columns={
    "packaging_en":              "packaging",
    "additives_en":              "additives",
    "ingredients_tags":          "ingredients",
    "manufacturing_places_tags": "manufacturing_places",
})

print(f"\n{'─' * _W}")
print(f"  Zwischenstand: {df.shape[1]} Spalten gesamt  |  {len(df):,} Zeilen")
print(f"{'─' * _W}")


────────────────────────────────────────────────────────────────────
  Zwischenstand: 191 Spalten gesamt  |  4,501,283 Zeilen
────────────────────────────────────────────────────────────────────


In [93]:
# ███ 🟢 VERÄNDERT DEN DATENSATZ (df) ███
# NOVA-Vorbereitung — KEIN Neuladen: wir arbeiten auf DEMSELBEN df wie die Bereinigung.
# Die fuer das Marker-Matching noetigen TAG-Spalten sind hier:
#   categories -> categories_tags (roh geladen) · ingredients -> "ingredients" (gereinigte Tags)
#   additives  -> additives_tags  (roh behalten)
# categories_tags + additives_tags noch normalisieren (lowercase, Komma-Abstaende; Praefixe bleiben).
# nova_group wurde bereits in Teil B auf 1-4/Int64 gesaeubert.
for c in ["categories_tags", "additives_tags"]:
    df[c] = clean_series(df[c]).str.lower().str.replace(r"\s*,\s*", ",", regex=True)

print(f"Zeilen gesamt        : {len(df):,}")
print(f"nova_group offiziell : {df['nova_group'].notna().sum():,}  ({df['nova_group'].notna().mean()*100:.1f} %)")
for tt, c in [("categories", "categories_tags"), ("ingredients", "ingredients"), ("additives", "additives_tags")]:
    print(f"{c:16s} : {df[c].notna().sum():,}  ({df[c].notna().mean()*100:.1f} %)")

Zeilen gesamt        : 4,501,283
nova_group offiziell : 1,129,023  (25.1 %)
categories_tags  : 1,848,143  (41.1 %)
ingredients      : 1,277,639  (28.4 %)
additives_tags   : 697,070  (15.5 %)


In [94]:
# ⚙️ SETUP / DEFINITIONEN — verändert df NICHT
# NOVA-Marker, 1:1 portiert aus Open Food Facts (Stand 2026-06-21):
#   * lib/ProductOpener/Config_off.pm  ->  $options{nova_groups_tags}   (Kern, inkl. Additive je E-Nummer)
#   * taxonomies/food/{categories,ingredients}.txt  ->  nova:en: Eigenschaft (zusaetzliche Marker)
# Schluessel = "tagtype/tagid", Wert = NOVA-Gruppe (2/3/4). Bei Konflikt wurde die hoehere Gruppe behalten.
# Hinweis: Additiv-KLASSEN-Marker in der Taxonomie sind dort auskommentiert/inaktiv -> Additive
# werden (wie im echten OFF) direkt ueber die E-Nummer gematcht, KEIN Klassen-Umweg.
NOVA_MARKER = {
    # --- categories (30) ---
    "categories/en:alcoholic-beverages": 3, "categories/en:animal-fats": 2, "categories/en:baby-milks": 3,
    "categories/en:beers": 3, "categories/en:candies": 3, "categories/en:cheeses": 3,
    "categories/en:chocolates": 3, "categories/en:ciders": 3, "categories/en:fats": 2,
    "categories/en:hard-liquors": 3, "categories/en:honeys": 2, "categories/en:ice-creams": 3,
    "categories/en:maple-syrups": 2, "categories/en:meal-kits": 3, "categories/en:meals": 3,
    "categories/en:pates": 3, "categories/en:prepared-meats": 3, "categories/en:salts": 2,
    "categories/en:salty-snacks": 3, "categories/en:sandwiches": 3, "categories/en:sausages": 3,
    "categories/en:sodas": 3, "categories/en:starches": 2, "categories/en:sugars": 2,
    "categories/en:sugary-snacks": 3, "categories/en:sweet-snacks": 3, "categories/en:terrines": 3,
    "categories/en:tofu": 3, "categories/en:vinegars": 2, "categories/en:wines": 3,
    # --- ingredients (57) ---
    "ingredients/en:anti-caking-agent": 3, "ingredients/en:anti-foaming-agent": 4, "ingredients/en:bread": 3,
    "ingredients/en:bulking-agent": 4, "ingredients/en:butter": 3, "ingredients/en:carbonating-agent": 4,
    "ingredients/en:casein": 4, "ingredients/en:cheese": 3, "ingredients/en:colour": 4,
    "ingredients/en:colour-stabilizer": 4, "ingredients/en:concentrated-whey-protein": 4,
    "ingredients/en:dextrose": 4, "ingredients/en:emulsifier": 4, "ingredients/en:firming-agent": 4,
    "ingredients/en:flavour": 4, "ingredients/en:flavour-enhancer": 4, "ingredients/en:flavouring": 4,
    "ingredients/en:fructose": 4, "ingredients/en:fruit-juice-concentrate": 4,
    "ingredients/en:gelling-agent": 4, "ingredients/en:glazing-agent": 4, "ingredients/en:glucose": 4,
    "ingredients/en:glucose-syrup": 4, "ingredients/en:gluten": 4,
    "ingredients/en:high-fructose-corn-syrup": 4, "ingredients/en:honey": 3, "ingredients/en:humectant": 4,
    "ingredients/en:hydrogenated-fat": 4, "ingredients/en:hydrogenated-oil": 4,
    "ingredients/en:hydrolysed-cereal": 4, "ingredients/en:hydrolysed-proteins": 4,
    "ingredients/en:invert-sugar": 4, "ingredients/en:lactose": 4, "ingredients/en:lecithin": 4,
    "ingredients/en:maltodextrin": 4, "ingredients/en:maple-syrup": 3,
    "ingredients/en:mechanically-separated-meat": 4, "ingredients/en:milk-powder": 3,
    "ingredients/en:milk-proteins": 4, "ingredients/en:modified-flour": 4,
    "ingredients/en:modified-starch": 4, "ingredients/en:preservative": 3, "ingredients/en:salt": 3,
    "ingredients/en:sauce": 3, "ingredients/en:sequestrant": 4, "ingredients/en:soy-preparation": 4,
    "ingredients/en:starch": 3, "ingredients/en:sugar": 3, "ingredients/en:sweetener": 4,
    "ingredients/en:thickener": 4, "ingredients/en:vegetable-fat": 3, "ingredients/en:vegetable-fiber": 4,
    "ingredients/en:vegetable-oil": 3, "ingredients/en:vegetal-oil": 3, "ingredients/en:whey": 4,
    "ingredients/en:whey-product": 4, "ingredients/en:whey-proteins": 4,
    # --- additives (240) ---
    "additives/en:e100": 4, "additives/en:e101": 4, "additives/en:e101a": 4, "additives/en:e102": 4,
    "additives/en:e103": 4, "additives/en:e104": 4, "additives/en:e105": 4, "additives/en:e106": 4,
    "additives/en:e107": 4, "additives/en:e110": 4, "additives/en:e1104": 4, "additives/en:e111": 4,
    "additives/en:e120": 4, "additives/en:e121": 4, "additives/en:e122": 4, "additives/en:e123": 4,
    "additives/en:e124": 4, "additives/en:e125": 4, "additives/en:e126": 4, "additives/en:e127": 4,
    "additives/en:e128": 4, "additives/en:e129": 4, "additives/en:e130": 4, "additives/en:e131": 4,
    "additives/en:e132": 4, "additives/en:e133": 4, "additives/en:e140": 4, "additives/en:e1400": 4,
    "additives/en:e1401": 4, "additives/en:e1402": 4, "additives/en:e1403": 4, "additives/en:e1404": 4,
    "additives/en:e1405": 4, "additives/en:e141": 4, "additives/en:e1410": 4, "additives/en:e1412": 4,
    "additives/en:e1413": 4, "additives/en:e1414": 4, "additives/en:e142": 4, "additives/en:e1420": 4,
    "additives/en:e1422": 4, "additives/en:e143": 4, "additives/en:e1440": 4, "additives/en:e1442": 4,
    "additives/en:e1450": 4, "additives/en:e1451": 4, "additives/en:e14xx": 4, "additives/en:e150": 4,
    "additives/en:e1505": 4, "additives/en:e150a": 4, "additives/en:e150b": 4, "additives/en:e150c": 4,
    "additives/en:e150d": 4, "additives/en:e151": 4, "additives/en:e152": 4, "additives/en:e1521": 4,
    "additives/en:e153": 4, "additives/en:e154": 4, "additives/en:e155": 4, "additives/en:e15x": 4,
    "additives/en:e160": 4, "additives/en:e160a": 4, "additives/en:e160b": 4, "additives/en:e160c": 4,
    "additives/en:e160d": 4, "additives/en:e160e": 4, "additives/en:e160f": 4, "additives/en:e161": 4,
    "additives/en:e161a": 4, "additives/en:e161b": 4, "additives/en:e161c": 4, "additives/en:e161d": 4,
    "additives/en:e161e": 4, "additives/en:e161f": 4, "additives/en:e161g": 4, "additives/en:e161h": 4,
    "additives/en:e161i": 4, "additives/en:e161j": 4, "additives/en:e162": 4, "additives/en:e163": 4,
    "additives/en:e163a": 4, "additives/en:e163b": 4, "additives/en:e163c": 4, "additives/en:e163d": 4,
    "additives/en:e163e": 4, "additives/en:e163f": 4, "additives/en:e164": 4, "additives/en:e165": 4,
    "additives/en:e166": 4, "additives/en:e170": 4, "additives/en:e171": 4, "additives/en:e172": 4,
    "additives/en:e173": 4, "additives/en:e174": 4, "additives/en:e175": 4, "additives/en:e180": 4,
    "additives/en:e181": 4, "additives/en:e182": 4, "additives/en:e202": 3, "additives/en:e249": 3,
    "additives/en:e250": 3, "additives/en:e251": 3, "additives/en:e252": 3, "additives/en:e290": 4,
    "additives/en:e322": 4, "additives/en:e325": 4, "additives/en:e326": 4, "additives/en:e327": 4,
    "additives/en:e328": 4, "additives/en:e329": 4, "additives/en:e400": 4, "additives/en:e401": 4,
    "additives/en:e402": 4, "additives/en:e403": 4, "additives/en:e404": 4, "additives/en:e405": 4,
    "additives/en:e406": 4, "additives/en:e407": 4, "additives/en:e407a": 4, "additives/en:e409": 4,
    "additives/en:e410": 4, "additives/en:e412": 4, "additives/en:e413": 4, "additives/en:e414": 4,
    "additives/en:e415": 4, "additives/en:e416": 4, "additives/en:e417": 4, "additives/en:e418": 4,
    "additives/en:e420": 4, "additives/en:e421": 4, "additives/en:e422": 4, "additives/en:e425": 4,
    "additives/en:e428": 4, "additives/en:e430": 4, "additives/en:e431": 4, "additives/en:e432": 4,
    "additives/en:e433": 4, "additives/en:e434": 4, "additives/en:e435": 4, "additives/en:e436": 4,
    "additives/en:e440": 4, "additives/en:e441": 4, "additives/en:e442": 4, "additives/en:e443": 4,
    "additives/en:e444": 4, "additives/en:e445": 4, "additives/en:e450": 4, "additives/en:e451": 4,
    "additives/en:e452": 4, "additives/en:e459": 4, "additives/en:e460": 4, "additives/en:e461": 4,
    "additives/en:e463": 4, "additives/en:e464": 4, "additives/en:e465": 4, "additives/en:e466": 4,
    "additives/en:e468": 4, "additives/en:e469": 4, "additives/en:e470": 4, "additives/en:e470a": 4,
    "additives/en:e470b": 4, "additives/en:e471": 4, "additives/en:e472a": 4, "additives/en:e472b": 4,
    "additives/en:e472c": 4, "additives/en:e472d": 4, "additives/en:e472e": 4, "additives/en:e472f": 4,
    "additives/en:e473": 4, "additives/en:e474": 4, "additives/en:e475": 4, "additives/en:e476": 4,
    "additives/en:e477": 4, "additives/en:e478": 4, "additives/en:e479b": 4, "additives/en:e480": 4,
    "additives/en:e481": 4, "additives/en:e482": 4, "additives/en:e483": 4, "additives/en:e491": 4,
    "additives/en:e492": 4, "additives/en:e493": 4, "additives/en:e494": 4, "additives/en:e495": 4,
    "additives/en:e551": 4, "additives/en:e620": 4, "additives/en:e621": 4, "additives/en:e622": 4,
    "additives/en:e623": 4, "additives/en:e624": 4, "additives/en:e625": 4, "additives/en:e626": 4,
    "additives/en:e627": 4, "additives/en:e628": 4, "additives/en:e629": 4, "additives/en:e630": 4,
    "additives/en:e631": 4, "additives/en:e632": 4, "additives/en:e633": 4, "additives/en:e634": 4,
    "additives/en:e635": 4, "additives/en:e636": 4, "additives/en:e637": 4, "additives/en:e640": 4,
    "additives/en:e641": 4, "additives/en:e650": 4, "additives/en:e900": 4, "additives/en:e900a": 4,
    "additives/en:e901": 4, "additives/en:e902": 4, "additives/en:e903": 4, "additives/en:e904": 4,
    "additives/en:e905": 4, "additives/en:e905c": 4, "additives/en:e905d": 4, "additives/en:e907": 4,
    "additives/en:e938": 4, "additives/en:e939": 4, "additives/en:e941": 4, "additives/en:e942": 4,
    "additives/en:e943a": 4, "additives/en:e943b": 4, "additives/en:e950": 4, "additives/en:e951": 4,
    "additives/en:e952": 4, "additives/en:e953": 4, "additives/en:e954": 4, "additives/en:e955": 4,
    "additives/en:e956": 4, "additives/en:e957": 4, "additives/en:e959": 4, "additives/en:e960": 4,
    "additives/en:e961": 4, "additives/en:e962": 4, "additives/en:e964": 4, "additives/en:e965": 4,
    "additives/en:e966": 4, "additives/en:e967": 4, "additives/en:e968": 4, "additives/en:e969": 4,
}

print("Marker gesamt:", len(NOVA_MARKER), dict(Counter(k.split("/")[0] for k in NOVA_MARKER)))
print("Gruppen-Verteilung der Marker:", dict(Counter(NOVA_MARKER.values())))

Marker gesamt: 327 {'categories': 30, 'ingredients': 57, 'additives': 240}
Gruppen-Verteilung der Marker: {3: 42, 2: 8, 4: 277}


In [95]:
# ⚙️ SETUP / DEFINITIONEN — verändert df NICHT
# ── Klassifikator: treuer Port von compute_nova_group (Food.pm) ──────────────────────────────
# In diesem (vereinigten) df heisst die gereinigte Zutaten-Tag-Spalte "ingredients".
NOVA_COLS = {"categories": "categories_tags", "ingredients": "ingredients", "additives": "additives_tags"}

# Start bei Gruppe 1; hoechste getroffene Markergruppe gewinnt, ABER eine Gruppe-2-Markierung
# wird NICHT von Gruppe 3 hochgestuft ("Zucker bleibt Gruppe 2"). 4 schlaegt immer alles.
def entscheide(groups):
    if 4 in groups: return 4
    if 2 in groups: return 2          # Gruppe-2 vor Gruppe-3 geschuetzt (Food.pm-Regel)
    if 3 in groups: return 3
    return 1                           # Gates erfuellt, aber kein Marker -> minimal verarbeitet

def _matched(frame, tagtype):
    s = frame[NOVA_COLS[tagtype]].dropna().str.split(",").explode().str.strip()
    g = (tagtype + "/" + s).map(NOVA_MARKER)          # dict-map (schnell); NaN wenn kein Marker
    return g.dropna().astype(int)

def klassifiziere(frame):
    # frame muss bereits die Gates erfuellen (Zutaten + Kategorie vorhanden, kein Non-Food)
    mg = pd.concat([_matched(frame, "categories"), _matched(frame, "ingredients"), _matched(frame, "additives")])
    gp = mg.groupby(level=0).agg(lambda x: set(x))     # Menge getroffener Gruppen je Produkt
    res = gp.reindex(frame.index).apply(lambda s: entscheide(s) if isinstance(s, set) else 1)
    return res.astype("Int64")

def gate(frame):
    # Abstinenz-Gates (Food.pm): Zutaten UND Kategorie vorhanden, kein Non-Food.
    # (OFF-Ausnahmen "Wasser/Gruppe-2 ohne Zutaten" und ">=50% unbekannte Zutaten" bewusst nicht repliziert.)
    has_ing = frame[NOVA_COLS["ingredients"]].notna()
    has_cat = frame[NOVA_COLS["categories"]].notna()
    nonfood = frame[NOVA_COLS["categories"]].str.contains("en:non-food-products", regex=False).fillna(False)
    return (has_ing & has_cat & ~nonfood).astype(bool)

print("Klassifikator + Gates definiert.")

Klassifikator + Gates definiert.


In [96]:
# ⚙️ SETUP / DEFINITIONEN — verändert df NICHT
# ↳ berechnet zudem das Auslieferungs-Gate `allowed`/`kill_a` für die nächste Zelle — NICHT entfernen
# ── Validierung gegen die offiziell gelabelten Zeilen (OBERGRENZE der Guete) ─────────────────
# Wichtig: diese Zeilen haben sauberere Eingaben als die Zielprodukte -> Accuracy hier ist ein
# optimistischer Oberwert.
labeled = df[df["nova_group"].notna()].copy()
g_lab   = gate(labeled)
val     = labeled[g_lab].copy()

y_true = val["nova_group"].astype(int)
y_pred = klassifiziere(val).astype(int)

print(f"Gelabelte Zeilen           : {len(labeled):,}")
print(f"davon Gates erfuellt (bewertbar): {int(g_lab.sum()):,}")
print(f"abstiniert (Gate gerissen)  : {int((~g_lab).sum()):,}  ({(~g_lab).mean()*100:.1f} %)\n")

konf = pd.crosstab(y_true, y_pred, rownames=["offiziell"], colnames=["abgeleitet"], dropna=False)
print("Konfusionsmatrix (Zeile = offiziell, Spalte = abgeleitet):")
print(konf, "\n")

acc      = (y_true == y_pred).mean()
mode_cls = int(y_true.mode().iloc[0])
baseline = (y_true == mode_cls).mean()
print(f"Accuracy : {acc*100:.1f} %   |   Baseline (immer {mode_cls}): {baseline*100:.1f} %   |   Delta: {(acc-baseline)*100:+.1f} pp\n")

prec, rec = {}, {}
for k in [1, 2, 3, 4]:
    tp = int(((y_pred == k) & (y_true == k)).sum())
    prec[k] = tp / max(int((y_pred == k).sum()), 1)
    rec[k]  = tp / max(int((y_true == k).sum()), 1)
print("Precision / Recall je Gruppe:")
print(pd.DataFrame({"precision": prec, "recall": rec}).round(3), "\n")

# ── Kill-Bedingung (VORAB festgelegt) ────────────────────────────────────────────────────────
kill_a  = (acc - baseline) >= 0.10                       # (a) Accuracy muss Baseline um >=10pp schlagen
allowed = {k for k in [1, 2, 3, 4] if prec[k] >= 0.80}   # (b) nur Gruppen mit Precision >=80% ausliefern
print(f"Kill (a) Accuracy >= Baseline+10pp : {'OK' if kill_a else 'GERISSEN -> keine Auslieferung'}")
print(f"Kill (b) auslieferbare Gruppen (Precision>=80%): {sorted(allowed) if kill_a else '— (a gerissen)'}")

Gelabelte Zeilen           : 1,129,023
davon Gates erfuellt (bewertbar): 955,136
abstiniert (Gate gerissen)  : 173,887  (15.4 %)

Konfusionsmatrix (Zeile = offiziell, Spalte = abgeleitet):
abgeleitet       1      2       3       4
offiziell                                
1           127325      0       0      13
2                0  19860       0       0
3              700      0  206844      86
4               39      9     464  599796 

Accuracy : 99.9 %   |   Baseline (immer 4): 62.9 %   |   Delta: +37.0 pp

Precision / Recall je Gruppe:
   precision  recall
1      0.994   1.000
2      1.000   1.000
3      0.998   0.996
4      1.000   0.999 

Kill (a) Accuracy >= Baseline+10pp : OK
Kill (b) auslieferbare Gruppen (Precision>=80%): [1, 2, 3, 4]


In [97]:
# ███ 🟢 VERÄNDERT DEN DATENSATZ (df) ███
# ── Anwenden auf die Kandidaten (nova fehlt + Gates erfuellt) + zwei neue Spalten ────────────
df["nova_group_derived"] = pd.array([pd.NA] * len(df), dtype="Int64")
df["nova_group_source"]  = pd.Series(pd.NA, index=df.index, dtype="string")
df.loc[df["nova_group"].notna(), "nova_group_source"] = "off_precomputed"

cand_mask = df["nova_group"].isna() & gate(df)
print(f"Kandidaten (nova NaN + Gates erfuellt): {int(cand_mask.sum()):,}")

if kill_a and allowed:
    pred = klassifiziere(df[cand_mask])
    pred = pred.where(pred.isin(list(allowed)), pd.NA).astype("Int64")   # nur Precision>=80%-Gruppen
    df.loc[cand_mask, "nova_group_derived"] = pred
    got = df["nova_group_derived"].notna()
    df.loc[got, "nova_group_source"] = "rule_derived"
    print(f"davon abgeleitet & ausgeliefert (Gruppen {sorted(allowed)}): {int(got.sum()):,}")
    print("\nVerteilung abgeleitete Gruppen:")
    print(df.loc[got, "nova_group_derived"].value_counts().sort_index())
else:
    print("Kill-Bedingung (a) gerissen -> KEINE abgeleiteten Werte ausgeliefert (nur Diagnose).")

# Konsistenz-Checks + Gesamtabdeckung
print("\nQuelle-Verteilung (nova_group_source):")
print(df["nova_group_source"].value_counts(dropna=False))
assert (df["nova_group_source"] == "off_precomputed").sum() == int(df["nova_group"].notna().sum())
assert df.loc[df["nova_group_source"] == "rule_derived", "nova_group"].isna().all()
merged = df["nova_group"].fillna(df["nova_group_derived"])
print(f"\nNOVA-Abdeckung: offiziell {df['nova_group'].notna().mean()*100:.1f} %  ->  inkl. abgeleitet {merged.notna().mean()*100:.1f} %")

# Stichprobe je abgeleiteter Gruppe
pd.set_option("display.max_colwidth", 70)
samp = df[df["nova_group_source"] == "rule_derived"]
beispiele = pd.concat([samp[samp["nova_group_derived"] == k].head(3) for k in [1, 2, 3, 4]])
beispiele[["product_name", "categories_tags", "ingredients", "additives_tags", "nova_group_derived","nova_group_source"]]

Kandidaten (nova NaN + Gates erfuellt): 117,297
davon abgeleitet & ausgeliefert (Gruppen [1, 2, 3, 4]): 117,297

Verteilung abgeleitete Gruppen:
nova_group_derived
1    80278
2     5598
3    31395
4       26
Name: count, dtype: Int64

Quelle-Verteilung (nova_group_source):
nova_group_source
<NA>               3254963
off_precomputed    1129023
rule_derived        117297
Name: count, dtype: Int64

NOVA-Abdeckung: offiziell 25.1 %  ->  inkl. abgeleitet 27.7 %


,product_name,categories_tags,ingredients,additives_tags,nova_group_derived,nova_group_source
12,granola Bio le Chocolaté,"en:plant-based-foods-and-beverages,en:plant-based-foods,en:fruits-...","es:honig-stillende-frauen-nicht-geeignet,es:d-bestrahlung-vermeide...",<NA>,1,rule_derived
31,Volle yoghurt,"en:beverages-and-beverages-preparations,en:beverages",en:nik-odżywczy-wspiera-prawidłowe-funkcjonowanie-organizmu-zeskan...,<NA>,1,rule_derived
69,NaN,"en:beverages-and-beverages-preparations,en:plant-based-foods-and-b...","es:fullstoff,es:zinkglu,es:conat,es:modifizierte-starke,en:biotin,...",<NA>,1,rule_derived
70,NaN,en:fats,"en:kakaomasse,en:rohrzucker,en:karamellisierte-kokosflocken,en:kak...",<NA>,2,rule_derived
296,huile végétale,"en:plant-based-foods-and-beverages,en:plant-based-foods,en:fats,en...",fr:total-des,<NA>,2,rule_derived
1033,คุกกี้สเปลท์เนยสดผสมข้าวกล้องงอก,"en:dairies,en:snacks,en:fats,en:spreads,en:sweet-snacks,en:biscuit...",th:แป้งสเปลท์ออร์แกนิค-ข้าวกล้องงอก-เนยสด-ไข่ไก่-ไอซ่ง-ผงฟู-กลิ่นว...,<NA>,2,rule_derived
18,Powdered peanut butter,"en:snacks,en:meals,en:rice-dishes,en:risottos,en:powder-peanut-butter","en:water,en:leptospermum-scoparium-mel,en:sodium-c14-c16-olefin-su...",<NA>,3,rule_derived
33,Anthony's Organic Cocoa Butter Chunks,"en:sandwiches,en:wraps",en:allulose,<NA>,3,rule_derived
48,Gummie,"en:meals,en:pasta-dishes,en:stuffed-pastas,en:ravioli,en:japanese-...","en:acrylate-adhesive,en:polyester,en:silicone-adhesive",<NA>,3,rule_derived
341652,Enduit à Cuisson Antiadhésif Original,en:non-open-products-facts,"en:canola-oil,en:oil-and-fat,en:vegetable-oil-and-fat,en:rapeseed-...","en:e322,en:e322i,en:e943b",4,rule_derived


In [98]:
df.info(verbose=True, show_counts=True)

<class 'pandas.DataFrame'>
Index: 4501283 entries, 0 to 4501354
Data columns (total 193 columns):
 #    Column                            Non-Null Count    Dtype  
---   ------                            --------------    -----  
 0    code                              4501283 non-null  str    
 1    url                               4501283 non-null  str    
 2    creator                           4501277 non-null  str    
 3    created_t                         4501283 non-null  int64  
 4    created_datetime                  4501283 non-null  str    
 5    last_modified_t                   4501283 non-null  int64  
 6    last_modified_datetime            4501283 non-null  str    
 7    last_modified_by                  4362722 non-null  str    
 8    last_updated_t                    4501109 non-null  float64
 9    last_updated_datetime             4501109 non-null  str    
 10   product_name                      4166147 non-null  str    
 11   abbreviated_product_name          2936

In [99]:
meine_spalten = ["brands", "brands_tags", "brands_en", 
                 "labels", "labels_tags", "labels_en", 
                 "countries", "countries_tags", "countries_en", 
                 "food_groups", "food_groups_tags", "food_groups_en", 
                 "main_category", "main_category_en", 
                 "pnns_groups_1", "pnns_groups_2",
                 "brand_owner",
                 "popularity_tags",
                 "origins", "origins_tags", "origins_en",
                 "purchase_places",
                 "cities_tags",
                 "no_nutrition_data", "brands_clean", "main_category_clean"]

def remove_duplicates(text):
    # Überspringe fehlende Werte (NaN)
    if pd.isna(text):
        return text
    
    # 1. Am Komma trennen und überflüssige Leerzeichen (strip) entfernen
    items = [item.strip() for item in str(text).split(',')]
    
    # 2. Duplikate entfernen (dict.fromkeys erhält im Gegensatz zu set() die ursprüngliche Reihenfolge)
    unique_items = list(dict.fromkeys(items))
    
    # 3. Wieder zu einem sauberen String zusammensetzen
    return ', '.join(unique_items)

vorhandene_spalten = [spalte for spalte in meine_spalten if spalte in df.columns]

for spalte in vorhandene_spalten:
    df[spalte] = df[spalte].apply(remove_duplicates)


In [100]:
df.to_csv(
    'openfoodfacts_bereinigt.csv.zip', 
    index=False, 
    sep='\t', 
    encoding='utf-8',
    compression="zip",
)